# SQL Interview Questions for Data Engineering

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*

This notebook is a hands-on bank of the SQL questions that show up in data-engineering interviews. Everything
runs against a small **in-memory DuckDB** database, so there is nothing to configure and no files are written.
It is deliberately granular: every idea gets its own explanation and its own small cell, so you can stop,
predict the output, and only then run the cell.

**How to run it**

1. Run the cells top to bottom with `Shift+Enter`, or use *Run All*.
2. The in-memory `duckdb` database is created by the setup cells near the top, so run those first; then read
   each question, predict the answer, and only then run its cell.
3. The only dependency is `duckdb` (already in the repository's `notebooks` extra), so the asserts double as
   CI checks via `make notebooks-check`.
4. This notebook is also live in the interactive viewer: serve the app with `make dev-local` and open
   `/topics/03_sql_interview_questions` - every cell executes against the site's persistent Python kernel.


**How to use it**

1. Run the cells in order (`Shift+Enter`), or use *Run All*.
2. Try to predict each result before you run the cell, then read the verification `assert`s.
3. Outputs are saved in the notebook, so it reads like a document without a kernel.
4. The last sections are the DuckDB-only skills (Part 7) and a bank of theory questions with model answers.


The SQL uses the analytics features DuckDB was built for: window functions, CTEs, `FILTER`, `LAG`/`LEAD` and
`string_agg ... ORDER BY` - plus `QUALIFY`, `PIVOT`/`UNPIVOT`, `ASOF JOIN`, typed `LIST`/`STRUCT` values and
readable `EXPLAIN` / `EXPLAIN ANALYZE` plans (Part 7). DuckDB hands results back as Python-native types
(`int`, `float`, `decimal.Decimal`, `datetime.date`), and every result below is deterministic - ties are
broken explicitly.

Companion notebooks: `01_python_basics.ipynb` (Python), `02_python_coding_problems.ipynb` (algorithms),
`04_pyspark_concepts.ipynb` (Spark), `05_interview_questions.ipynb` (mixed question bank).


## Setup: an in-memory DuckDB database

The setup is four small steps: open a connection, define two helpers, create and seed six tables, then check the
seed. Six tables keep the examples realistic:

| table | grain | notable columns |
| --- | --- | --- |
| `departments` | one row per department | `dept_id`, `dept_name`, `location` |
| `employees` | one row per employee | `manager_id` (self-reference), `salary`, `hire_date` (`DATE`) |
| `employee_updates` | append-only history | `emp_id`, `salary`, `updated_at` (several rows per employee) |
| `customers` | one row per customer | `region`, `signup_date` (`DATE`) |
| `orders` | one row per order | `status`, `amount`, `order_date` (`DATE`) |
| `events` | one row per event | `user_id`, `event_ts` (`TIMESTAMP`), `event_type` |


### The connection

`duckdb.connect(":memory:")` opens a private, throwaway database that lives only for this kernel session, so
re-running the notebook always starts from the same clean state. DuckDB is a columnar, vectorised analytics
engine: it runs plain SQL, aggregates by columns rather than rows, and - because it is in-process - needs no
server, no files and no configuration.


In [ ]:
import duckdb
from datetime import date, datetime

conn = duckdb.connect(":memory:")
print("duckdb version:", duckdb.__version__)


### Helper: `run`

`run(sql, params=())` executes one statement with optional bind parameters and returns every row as a list of
tuples. It is the helper the verification cells use, which keeps the `assert`s short and readable. Mutating
statements (`INSERT`/`UPDATE`/`DELETE`) come back with their affected-row count as a single row, which keeps
seeds and delete checks readable.


In [ ]:
def run(sql, params=()):
    """Execute one SQL statement and return all rows as a list of tuples."""
    return conn.execute(sql, params).fetchall()


### Helper: `show`

`show(sql, params=(), max_rows=30)` executes one query, prints it as a compact ASCII table, and also returns
the rows. It formats straight from DuckDB's own result rows (no pandas needed), so the answer is visible in
the saved output instead of only existing inside an `assert`.


In [ ]:
def show(sql, params=(), max_rows=30):
    """Print the result of one query as a small table, and return the rows."""
    result = conn.execute(sql, params)
    columns = [description[0] for description in result.description]
    rows = result.fetchall()
    rendered = [[str(value) for value in row] for row in rows]
    widths = [len(name) for name in columns]
    for row in rendered[:max_rows]:
        for i, text in enumerate(row):
            widths[i] = max(widths[i], len(text))

    print(" | ".join(name.ljust(widths[i]) for i, name in enumerate(columns)))
    print("-+-".join("-" * width for width in widths))
    for row in rendered[:max_rows]:
        print(" | ".join(text.ljust(widths[i]) for i, text in enumerate(row)))
    if len(rows) > max_rows:
        print(f"... {len(rows) - max_rows} more rows")
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})")
    return rows


### Seed data: six tables

Each table below is created and seeded in its own cell. `employees.emp_id` 1-14 covers four departments, with
two departments that have no employees below the average, one tied salary pair (Bob and Carol at 120000) and
two duplicate email pairs. `customers` 7 and 8 have never ordered, which the anti-join questions rely on.


#### `departments` (4 rows)

The small dimension table. `dept_id` is the primary key, `dept_name` is unique in practice, and `location` is a
plain attribute. Every employee references one of these rows, so this table must be seeded first.

In [ ]:
run('''
CREATE TABLE departments (
    dept_id   INTEGER PRIMARY KEY,
    dept_name TEXT NOT NULL,
    location  TEXT NOT NULL
);
''')

run('''
INSERT INTO departments (dept_id, dept_name, location) VALUES
    (1, 'Engineering', 'New York'),
    (2, 'Sales',       'Chicago'),
    (3, 'Marketing',   'Austin'),
    (4, 'Support',     'Remote');
''')

show("SELECT * FROM departments ORDER BY dept_id")

#### `employees` (14 rows)

The workhorse table. `emp_id` is the primary key, `dept_id` references `departments`, and `manager_id` is a
**self-reference** to `employees.emp_id` - `NULL` marks the top of the hierarchy (Alice, Frank, Judy, Leo).
`salary` is an integer and `hire_date` is a real `DATE`, so date arithmetic and comparisons behave like a
warehouse. The self-referencing foreign key is real, and DuckDB checks FKs per statement against the last
committed state, so rows are seeded in hierarchy-depth order. `email` is deliberately *not* unique:
Dave/Kevin and Heidi/Nina share one.


In [ ]:
run('''
CREATE TABLE employees (
    emp_id     INTEGER PRIMARY KEY,
    name       TEXT NOT NULL,
    dept_id    INTEGER NOT NULL REFERENCES departments(dept_id),
    manager_id INTEGER REFERENCES employees(emp_id),
    salary     INTEGER NOT NULL,
    hire_date  DATE NOT NULL,
    email      TEXT NOT NULL
);
''')

# DuckDB checks foreign keys per statement (against the last committed
# table state), so self-references must be seeded in hierarchy depth order:
# the four managers, then their reports, then the reports of reports.
run('''
INSERT INTO employees (emp_id, name, dept_id, manager_id, salary, hire_date, email) VALUES
    (1,  'Alice',   1, NULL, 150000, '2018-01-15', 'alice@example.com'),
    (6,  'Frank',   2, NULL, 130000, '2017-05-20', 'frank@example.com'),
    (10, 'Judy',    3, NULL, 105000, '2019-11-01', 'judy@example.com'),
    (12, 'Leo',     4, NULL,  90000, '2018-08-08', 'leo@example.com');
''')

run('''
INSERT INTO employees (emp_id, name, dept_id, manager_id, salary, hire_date, email) VALUES
    (7,  'Grace',   2, 6,     98000, '2020-02-14', 'grace@example.com'),
    (8,  'Heidi',   2, 6,     87000, '2021-07-01', 'heidi@example.com'),
    (9,  'Ivan',    2, 6,     87000, '2022-03-15', 'ivan@example.com'),
    (11, 'Kevin',   3, 10,    88000, '2021-04-01', 'dave@example.com');
''')

run('''
INSERT INTO employees (emp_id, name, dept_id, manager_id, salary, hire_date, email) VALUES
    (2,  'Bob',     1, 1,    120000, '2019-03-01', 'bob@example.com'),
    (3,  'Carol',   1, 1,    120000, '2020-06-15', 'carol@example.com'),
    (13, 'Mallory', 4, 12,    72000, '2022-05-05', 'mallory@example.com'),
    (14, 'Nina',    4, 12,    68000, '2023-02-01', 'heidi@example.com');
''')

run('''
INSERT INTO employees (emp_id, name, dept_id, manager_id, salary, hire_date, email) VALUES
    (4,  'Dave',    1, 2,     95000, '2021-09-01', 'dave@example.com'),
    (5,  'Eve',     1, 2,     90000, '2022-01-10', 'eve@example.com');
''')

print("employees seeded:", run("SELECT COUNT(*) FROM employees")[0][0])

#### `employee_updates` (8 rows)

An **append-only history**: several rows per `emp_id`, each recording a new salary at an `updated_at` date.
There is no primary key, so DuckDB's physical `rowid` is the only per-row identifier - which is why the
deduplication questions use it as a tie-breaker. Only employees 1, 2, 3, 6 and 7 appear here.


In [ ]:
run('''
CREATE TABLE employee_updates (
    emp_id     INTEGER NOT NULL,
    dept_id    INTEGER NOT NULL,
    salary     INTEGER NOT NULL,
    updated_at DATE NOT NULL
);
''')

run('''
INSERT INTO employee_updates (emp_id, dept_id, salary, updated_at) VALUES
    (1, 1, 155000, '2023-01-01'),
    (1, 1, 165000, '2023-06-01'),
    (2, 1, 122000, '2023-02-01'),
    (2, 1, 128000, '2023-05-01'),
    (3, 1, 125000, '2023-03-01'),
    (6, 2, 135000, '2023-01-15'),
    (6, 2, 140000, '2023-07-01'),
    (7, 2, 101000, '2023-04-01');
''')

show("SELECT * FROM employee_updates ORDER BY emp_id, updated_at")

#### `customers` (8 rows)

One row per customer with a `region` and a `signup_date` (`DATE`). Customers 7 (`Gamma SA`) and 8 (`Hotel BV`)
never place an order, so they are the fixed answer to every "who has no orders" question.


In [ ]:
run('''
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    region      TEXT NOT NULL,
    signup_date DATE NOT NULL
);
''')

run('''
INSERT INTO customers (customer_id, name, region, signup_date) VALUES
    (1, 'Acme Corp',   'North', '2022-01-05'),
    (2, 'Beta LLC',    'North', '2022-02-10'),
    (3, 'Cyan Inc',    'South', '2022-03-15'),
    (4, 'Delta Co',    'South', '2022-04-20'),
    (5, 'Echo Ltd',    'West',  '2022-05-25'),
    (6, 'Foxtrot GmbH','West',  '2022-06-30'),
    (7, 'Gamma SA',    'East',  '2023-01-10'),
    (8, 'Hotel BV',    'East',  '2023-02-15');
''')

show("SELECT * FROM customers ORDER BY customer_id")

#### `orders` (14 rows)

One row per order. `customer_id` references `customers`, `status` is one of `completed`, `pending` or
`cancelled`, `amount` is an integer, and `order_date` is a real `DATE`. The mix of statuses matters: most
questions filter to `completed`, and the pivot question counts all three.


In [ ]:
run('''
CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
    status      TEXT NOT NULL,
    amount      INTEGER NOT NULL,
    order_date  DATE NOT NULL
);
''')

run('''
INSERT INTO orders (order_id, customer_id, status, amount, order_date) VALUES
    (1,  1, 'completed', 100, '2023-01-05'),
    (2,  1, 'completed', 200, '2023-02-10'),
    (3,  1, 'completed', 150, '2023-03-15'),
    (4,  2, 'completed', 300, '2023-01-20'),
    (5,  2, 'cancelled', 250, '2023-02-05'),
    (6,  2, 'completed', 400, '2023-03-20'),
    (7,  3, 'completed', 120, '2023-01-25'),
    (8,  3, 'completed', 180, '2023-02-28'),
    (9,  4, 'pending',   220, '2023-02-14'),
    (10, 4, 'completed', 320, '2023-03-30'),
    (11, 5, 'completed', 500, '2023-01-02'),
    (12, 5, 'completed', 450, '2023-03-05'),
    (13, 6, 'completed', 275, '2023-02-18'),
    (14, 6, 'cancelled', 325, '2023-03-25');
''')

show("SELECT * FROM orders ORDER BY order_id")

#### `events` (15 rows)

One row per user event with a real **`TIMESTAMP`** and an `event_type`. User 1 has consecutive daily logins
(a gaps-and-islands case), user 3 has tight timestamps then a 65-minute gap (a sessionization case), and users
2-4 are small enough to verify by hand.


In [ ]:
run('''
CREATE TABLE events (
    event_id   INTEGER PRIMARY KEY,
    user_id    INTEGER NOT NULL,
    event_ts   TIMESTAMP NOT NULL,
    event_type TEXT NOT NULL
);
''')

run('''
INSERT INTO events (event_id, user_id, event_ts, event_type) VALUES
    (1,  1, '2023-01-01 09:00:00', 'login'),
    (2,  1, '2023-01-02 09:05:00', 'login'),
    (3,  1, '2023-01-03 08:55:00', 'login'),
    (4,  1, '2023-01-05 09:00:00', 'login'),
    (5,  1, '2023-01-06 09:10:00', 'login'),
    (6,  2, '2023-01-01 10:00:00', 'login'),
    (7,  2, '2023-01-02 10:00:00', 'login'),
    (8,  2, '2023-01-04 10:00:00', 'login'),
    (9,  3, '2023-02-01 10:00:00', 'view'),
    (10, 3, '2023-02-01 10:05:00', 'click'),
    (11, 3, '2023-02-01 10:25:00', 'view'),
    (12, 3, '2023-02-01 11:30:00', 'purchase'),
    (13, 3, '2023-02-01 11:45:00', 'view'),
    (14, 4, '2023-02-01 10:00:00', 'login'),
    (15, 4, '2023-02-01 10:20:00', 'view');
''')

show("SELECT * FROM events ORDER BY event_id")

### Row-count sanity check

Before any question runs, confirm that every table holds exactly the number of rows it should. A broken seed
would otherwise surface as a confusing assertion failure twenty cells later.

In [ ]:
expected_counts = {
    "departments": 4,
    "employees": 14,
    "employee_updates": 8,
    "customers": 8,
    "orders": 14,
    "events": 15,
}

for table, expected in expected_counts.items():
    actual = run(f"SELECT COUNT(*) FROM {table}")[0][0]
    assert actual == expected, (table, actual, expected)
    print(f"{table:<17} {actual:>3} rows")

### Structural invariants

Shape checks that the questions depend on: the helpers return `list`s of tuples, a `DATE` column comes back as
a `datetime.date`, four managers sit at the top of the hierarchy, four departments are represented by
employees, and exactly two customers have never ordered.


In [ ]:
rows = run("SELECT dept_id, dept_name FROM departments ORDER BY dept_id")
assert isinstance(rows, list) and all(isinstance(row, tuple) for row in rows)
assert rows == [(1, "Engineering"), (2, "Sales"), (3, "Marketing"), (4, "Support")]
assert len(show("SELECT customer_id FROM customers ORDER BY customer_id")) == 8

# DuckDB hands back Python-native types: a DATE column arrives as datetime.date, not a string
assert run("SELECT order_id, order_date FROM orders ORDER BY order_id LIMIT 1") == [(1, date(2023, 1, 5))]

assert run("SELECT COUNT(*) FROM employees WHERE manager_id IS NULL") == [(4,)]
assert run("SELECT COUNT(DISTINCT dept_id) FROM employees") == [(4,)]
assert run(
    """
    SELECT COUNT(*) FROM customers c
    WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
    """
) == [(2,)]
print("seed verified")


---
## Part 1 - Ranking and top-N

`ROW_NUMBER`, `RANK` and `DENSE_RANK` all number rows inside a `PARTITION BY`, but they break ties differently.
Almost every "top N per group" interview question is really a question about which one you need.

### Q1. Second-highest salary

Return the second-highest salary in `employees`. Repeated salaries count as one value.

**Expected result.** One row, `(130000,)`: Alice earns 150000 and Frank earns 130000, so 130000 is the runner-up.

In [ ]:
show("""
SELECT MAX(salary) AS second_highest
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees);
""")

**Check 1 - the subquery answer.** `MAX(salary)` below the overall maximum is the second-highest value, because
`<` excludes every copy of the top salary.

In [ ]:
assert run("""
SELECT MAX(salary) FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees)
""") == [(130000,)]
print("second-highest salary is 130000")

**Check 2 - the window-function alternative.** Rank *distinct* salaries in descending order and keep rank 2;
`DENSE_RANK` gives ties the same rank, so no salary can be duplicated.

In [ ]:
assert run("""
SELECT salary FROM (
    SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
    FROM employees
) WHERE rnk = 2
""") == [(130000,)]
print("DENSE_RANK agrees")

**Approach and pitfalls.** `MAX(salary) WHERE salary < (SELECT MAX(salary) ...)` is the classic one-liner. The
trap is the naive `ORDER BY salary DESC LIMIT 1 OFFSET 1`, which returns the same salary again when the top salary
is tied. Subtracting the maximum also degrades gracefully: if only one salary exists the result is `NULL` rather
than an error.

**Follow-ups.**

1. How would you return the whole row of the second-highest earner, not just the salary?
2. What does `DENSE_RANK` return instead of `NULL` when the table has a single salary?

### Q2. Nth-highest salary

Return the Nth-highest distinct salary, with N passed as a bind parameter. Here N = 3.

**Expected result.** One row, `(120000,)`. The distinct salaries in descending order start 150000, 130000,
120000 - and 120000 is shared by Bob and Carol, which is exactly where naive ranking implementations go wrong.

In [ ]:
n = 3  # the rank we want

show(
    """
    SELECT DISTINCT salary
    FROM employees
    ORDER BY salary DESC
    LIMIT 1 OFFSET ?;
    """,
    (n - 1,),
)

**Check 1 - the portable answer.** `SELECT DISTINCT salary ... LIMIT 1 OFFSET n-1` treats the ordered distinct
values as a zero-based list, so N = 3 means offset 2.

In [ ]:
n = 3
assert run(
    "SELECT DISTINCT salary FROM employees ORDER BY salary DESC LIMIT 1 OFFSET ?",
    (n - 1,),
) == [(120000,)]
print("Nth-highest via LIMIT/OFFSET:", 120000)

**Check 2 - the window answer.** `DENSE_RANK` assigns one rank per distinct salary, so `DISTINCT + rnk = n`
still yields exactly one value.

In [ ]:
assert run(
    """
    SELECT DISTINCT salary FROM (
        SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
        FROM employees
    ) WHERE rnk = ?
    """,
    (n,),
) == [(120000,)]
print("Nth-highest via DENSE_RANK:", 120000)

**Check 3 - why `DISTINCT` matters.** Dropping it exposes the tie: rank 3 holds two employees at 120000.

In [ ]:
assert run(
    """
    SELECT salary FROM (
        SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
        FROM employees
    ) WHERE rnk = ?
    """,
    (n,),
) == [(120000,), (120000,)]
print("rank 3 holds two tied rows")

**Approach and pitfalls.** `LIMIT 1 OFFSET n-1` over `SELECT DISTINCT salary` is portable and cheap. The window
version reads better in a longer query but must be wrapped in `DISTINCT` (or `MIN`) when ties exist, because
`DENSE_RANK` numbers ties with the same rank and therefore emits several rows. `ROW_NUMBER` never emits ties but
also never gives you a stable definition of "Nth".

**Follow-ups.**

1. What should the query return when N is larger than the number of distinct salaries?
2. Which index would make `ORDER BY salary DESC` avoid a sort?

### Q3. Top-N per group: ROW_NUMBER vs RANK vs DENSE_RANK

Return the top 3 employees by salary *per department*, then compare how many rows each ranking function keeps.

**Expected result.** The top-3 list has 11 rows (department 2 has only three slots filled because Heidi and Ivan
tie at 87000). The comparison shows `ROW_NUMBER` keeping exactly 3 per department, `RANK` keeping 4 for Sales, and
`DENSE_RANK` keeping 4 for Sales *and* 4 for Engineering - the tie at 120000 lets rank 3 exist there.

In [ ]:
show("""
SELECT dept_id, name, salary
FROM (
    SELECT dept_id, name, salary,
           ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC, emp_id) AS rn
    FROM employees
) ranked
WHERE rn <= 3
ORDER BY dept_id, salary DESC, name;
""")

**Check 1 - the top-3 list.** `ROW_NUMBER` with a deterministic tie-breaker (`emp_id`) picks exactly three rows
per department, and the result is stable across runs.

In [ ]:
n = 3
assert run(
    """
    SELECT dept_id, name, salary FROM (
        SELECT dept_id, name, salary,
               ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC, emp_id) AS rn
        FROM employees
    ) WHERE rn <= ?
    ORDER BY dept_id, salary DESC, name
    """,
    (n,),
) == [
    (1, "Alice", 150000), (1, "Bob", 120000), (1, "Carol", 120000),
    (2, "Frank", 130000), (2, "Grace", 98000), (2, "Heidi", 87000),
    (3, "Judy", 105000), (3, "Kevin", 88000),
    (4, "Leo", 90000), (4, "Mallory", 72000), (4, "Nina", 68000),
]
print("top-3 per department:", 11, "rows")

**Check 2 - how many rows each function keeps at the same threshold.** Count, per department, how many rows pass
`<= 3` under each ranking function.

In [ ]:
assert run(
    """
    SELECT dept_id,
           COUNT(CASE WHEN rn   <= 3 THEN 1 END) AS by_row_number,
           COUNT(CASE WHEN rnk  <= 3 THEN 1 END) AS by_rank,
           COUNT(CASE WHEN drnk <= 3 THEN 1 END) AS by_dense_rank
    FROM (
        SELECT dept_id, salary,
               ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC, emp_id) AS rn,
               RANK()       OVER (PARTITION BY dept_id ORDER BY salary DESC) AS rnk,
               DENSE_RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS drnk
        FROM employees
    )
    GROUP BY dept_id ORDER BY dept_id
    """
) == [(1, 3, 3, 4), (2, 3, 4, 4), (3, 2, 2, 2), (4, 3, 3, 3)]
print("ROW_NUMBER keeps 3, RANK/DENSE_RANK keep the ties")

**Check 3 - the mechanism, seen directly.** Department 1 has the 120000 tie, so `RANK` skips to 4 while
`DENSE_RANK` moves to 3. The `WINDOW w AS (...)` clause names the shared window definition once.

In [ ]:
assert run(
    """
    SELECT name, salary,
           ROW_NUMBER() OVER w AS rn,
           RANK()       OVER w AS rnk,
           DENSE_RANK() OVER w AS drnk
    FROM employees
    WHERE dept_id = 1
    WINDOW w AS (PARTITION BY dept_id ORDER BY salary DESC)
    ORDER BY salary DESC, emp_id
    """
) == [
    ("Alice", 150000, 1, 1, 1),
    ("Bob", 120000, 2, 2, 2),
    ("Carol", 120000, 3, 2, 2),
    ("Dave", 95000, 4, 4, 3),
    ("Eve", 90000, 5, 5, 4),
]
print("Q3 passed")

**Approach and pitfalls.** Filter on the ranking in an outer query - you cannot reference a window alias in the
same `WHERE` clause, because window functions are evaluated after `WHERE`. `ROW_NUMBER` is the right choice for
"exactly N rows per group"; use `RANK`/`DENSE_RANK` when ties should all survive (for example "everyone in the top
3 salaries"). Always add a deterministic tie-breaker such as `emp_id` to `ROW_NUMBER`'s `ORDER BY`, or the row you
keep is arbitrary and non-reproducible.

**Follow-ups.**

1. How would you return only the single highest-paid employee per department?
2. Why is `WHERE rn <= 3` allowed in the outer query but `WHERE ROW_NUMBER() OVER (...) <= 3` is not?

---
## Part 2 - Deduplication and history

Real tables arrive with duplicates: append-only history, retried loads, and dirty source systems. These questions
are about collapsing many rows per key into one, safely and reproducibly.

### Q4. Deduplicate, keeping the latest row

`employee_updates` is append-only: several rows per `emp_id`, each with an `updated_at` timestamp. Return one row
per employee - the most recent.

**Expected result.** Five rows: `(1, 1, 165000, '2023-06-01')`, `(2, 1, 128000, '2023-05-01')`,
`(3, 1, 125000, '2023-03-01')`, `(6, 2, 140000, '2023-07-01')`, `(7, 2, 101000, '2023-04-01')`.

In [ ]:
show("""
SELECT emp_id, dept_id, salary, updated_at
FROM (
    SELECT emp_id, dept_id, salary, updated_at,
           ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY updated_at DESC, rowid DESC) AS rn
    FROM employee_updates
) ranked
WHERE rn = 1
ORDER BY emp_id;
""")

**Check 1 - one row per employee.** Partition by `emp_id`, order by `updated_at` descending, and keep `rn = 1`.
`rowid DESC` breaks a timestamp tie deterministically.

In [ ]:
assert run(
    """
    SELECT emp_id, dept_id, salary, updated_at FROM (
        SELECT emp_id, dept_id, salary, updated_at,
               ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY updated_at DESC, rowid DESC) AS rn
        FROM employee_updates
    ) WHERE rn = 1 ORDER BY emp_id
    """
) == [
    (1, 1, 165000, date(2023, 6, 1)),
    (2, 1, 128000, date(2023, 5, 1)),
    (3, 1, 125000, date(2023, 3, 1)),
    (6, 2, 140000, date(2023, 7, 1)),
    (7, 2, 101000, date(2023, 4, 1)),
]
print("dedupe kept the latest row per employee")


**Check 2 - the `GROUP BY MAX` alternative.** Aggregate to the newest timestamp per employee, then rejoin to get
the whole row. It works here because no employee has two updates with the same timestamp.

In [ ]:
assert run(
    """
    SELECT u.emp_id, u.dept_id, u.salary, u.updated_at
    FROM employee_updates u
    JOIN (SELECT emp_id, MAX(updated_at) AS max_ts FROM employee_updates GROUP BY emp_id) m
      ON m.emp_id = u.emp_id AND m.max_ts = u.updated_at
    ORDER BY u.emp_id
    """
) == [
    (1, 1, 165000, date(2023, 6, 1)),
    (2, 1, 128000, date(2023, 5, 1)),
    (3, 1, 125000, date(2023, 3, 1)),
    (6, 2, 140000, date(2023, 7, 1)),
    (7, 2, 101000, date(2023, 4, 1)),
]
print("Q4 passed")

**Approach and pitfalls.** `ROW_NUMBER() ... ORDER BY updated_at DESC` inside a subquery, filtered by `rn = 1`
out in the outer query, is the general recipe. The tie-breaker (`rowid DESC` here) matters: without it, two rows
sharing the maximum timestamp give a non-deterministic answer. The `GROUP BY MAX` + self-join variant breaks when
two rows share the maximum timestamp - it returns both. For large tables, filtering to the latest partition avoids the shuffle, and DuckDB's `QUALIFY` clause drops
the subquery level entirely - Part 7, skill 1.

**Follow-ups.**

1. How would you keep the *first* row per key instead of the latest?
2. What changes if the dedupe key is the combination `(emp_id, dept_id)`?

### Q5. Delete duplicate rows in place

Physically delete the stale rows from a copy of `employee_updates`, keeping one row per `emp_id`. We work on a
scratch table so the seed data stays intact for the rest of the notebook.

**Expected result.** 8 rows before the `DELETE`, 5 after, with only the latest timestamp per employee remaining.

**The scratch copy.** `CREATE TABLE ... AS SELECT * FROM employee_updates` clones both the rows and the shape, so
the rest of the notebook still sees the original 8 rows.

In [ ]:
run("DROP TABLE IF EXISTS updates_scratch")
run("CREATE TABLE updates_scratch AS SELECT * FROM employee_updates")

print("before:", run("SELECT COUNT(*) FROM updates_scratch")[0][0], "rows")

**The delete.** Compute the surviving `rowid`s first (rank 1 per employee) and delete everything not in that
set. The subquery is materialised before any rows are removed, so the survivor set does not shrink mid-delete.
DuckDB also reports the affected-row count back to the client, so the delete itself is observable.


In [ ]:
deleted = run("""
DELETE FROM updates_scratch
WHERE rowid NOT IN (
    SELECT rowid FROM (
        SELECT rowid,
               ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY updated_at DESC, rowid DESC) AS rn
        FROM updates_scratch
    ) WHERE rn = 1
)
""")[0][0]

show("SELECT emp_id, salary, updated_at FROM updates_scratch ORDER BY emp_id")
print("deleted", deleted, "stale rows")


**Check 1 - five rows survive, one per employee.** Both the count and the surviving values are asserted, so a
delete that removes the wrong rows cannot pass.

In [ ]:
assert run("SELECT COUNT(*) FROM updates_scratch") == [(5,)]
assert run("SELECT emp_id, salary, updated_at FROM updates_scratch ORDER BY emp_id") == [
    (1, 165000, date(2023, 6, 1)),
    (2, 128000, date(2023, 5, 1)),
    (3, 125000, date(2023, 3, 1)),
    (6, 140000, date(2023, 7, 1)),
    (7, 101000, date(2023, 4, 1)),
]
print("scratch table now holds 5 rows")

**Check 2 - the same result without window functions.** A correlated `EXISTS` deletes a row when a strictly newer
version of the same employee exists, using `rowid` to break timestamp ties.

In [ ]:
run("DROP TABLE IF EXISTS updates_scratch2")
run("CREATE TABLE updates_scratch2 AS SELECT * FROM employee_updates")
run("""
DELETE FROM updates_scratch2
WHERE EXISTS (
    SELECT 1 FROM employee_updates keep
    WHERE keep.emp_id = updates_scratch2.emp_id
      AND (keep.updated_at > updates_scratch2.updated_at
           OR (keep.updated_at = updates_scratch2.updated_at AND keep.rowid > updates_scratch2.rowid))
)
""")
assert run("SELECT COUNT(*) FROM updates_scratch2") == [(5,)]
print("Q5 passed")

**Approach and pitfalls.** The safe pattern is "identify survivors, delete everything else" - expressed as
`rowid NOT IN (SELECT rowid FROM ... WHERE rn = 1)`. Two traps: (1) the parallel `DELETE FROM t WHERE t.ctid NOT
IN (SELECT MAX(ctid) ...)` style fails when the tie-breaker is not unique; (2) an unqualified subquery that reads
the table you are deleting from must see a stable snapshot - DuckDB materialises the survivor set before any rows
go, but engines that stream should be verified before you trust them. `rowid` is DuckDB's implicit physical row
id: fine as a within-statement tie-breaker, not a durable key. In production, do the delete inside a transaction,
verify the row count, and consider `CREATE TABLE ... AS SELECT` + swap when the table is large.


**Follow-ups.**

1. How would you delete duplicates but keep the row with the largest surrogate key?
2. Why is a `plain DELETE ... WHERE rowid IN (...)` often preferred to `NOT IN (...)`, and what happens if the
   subquery returns a `NULL`?

### Q6. Find duplicate emails

List every email address used by more than one employee.

**Expected result.** Two rows: `('dave@example.com', 2, 'Dave,Kevin')` and `('heidi@example.com', 2,
'Heidi,Nina')`. Everything else is unique.

In [ ]:
show("""
SELECT email,
       COUNT(*) AS copies,
       string_agg(name, ',' ORDER BY name) AS people
FROM employees
GROUP BY email
HAVING COUNT(*) > 1
ORDER BY email;
""")


**Check 1 - the duplicated addresses.** `GROUP BY email HAVING COUNT(*) > 1` is the whole answer, and
`GROUP_CONCAT ... ORDER BY` makes the companion names deterministic.

In [ ]:
assert run(
    """
    SELECT email, COUNT(*), string_agg(name, ',' ORDER BY name)
    FROM employees GROUP BY email HAVING COUNT(*) > 1 ORDER BY email
    """
) == [
    ("dave@example.com", 2, "Dave,Kevin"),
    ("heidi@example.com", 2, "Heidi,Nina"),
]
print("two duplicate emails found")


**Check 2 - the offending row ids.** This is the shape a cleanup job needs: the primary keys to rewrite, not the
aggregated count.

In [ ]:
assert run(
    """
    SELECT emp_id, email FROM employees
    WHERE email IN (SELECT email FROM employees GROUP BY email HAVING COUNT(*) > 1)
    ORDER BY email, emp_id
    """
) == [
    (4, "dave@example.com"), (11, "dave@example.com"),
    (8, "heidi@example.com"), (14, "heidi@example.com"),
]
print("Q6 passed")

**Approach and pitfalls.** `GROUP BY email HAVING COUNT(*) > 1` is the whole answer. The pitfall is case and
whitespace: `Dave@Example.com ` and `dave@example.com` are different strings but the same mailbox, so real
dedupe pipelines normalise first with `lower(trim(email))`. DuckDB collects the companion names with
`string_agg(name, ',' ORDER BY name)` - deterministic order; MySQL spells it
`GROUP_CONCAT(name ORDER BY name SEPARATOR ',')`, and DuckDB accepts `group_concat` as an alias.

**Follow-ups.**

1. How would you enforce uniqueness so the duplicate can never be inserted again?
2. How would you write the same query against a normalised email (lower-cased) column?

### Q7. UPDATE from another table

Apply the latest `employee_updates.salary` to a scratch copy of `employees`. Employees with no update row must be
left untouched.

**Expected result.** Exactly five salaries change: Alice 165000, Bob 128000, Carol 125000, Frank 140000 and Grace
101000. The other nine employees keep their original salaries.

**The scratch copy.** As in Q5, work on `employees_scratch` so `employees` keeps its seed values for the later
questions.

In [ ]:
run("DROP TABLE IF EXISTS employees_scratch")
run("CREATE TABLE employees_scratch AS SELECT * FROM employees")

print("copy created:", run("SELECT COUNT(*) FROM employees_scratch")[0][0], "employees")

**The update.** A correlated scalar subquery with `ORDER BY ... LIMIT 1` picks the newest salary per employee,
and `WHERE EXISTS` guards the rows that have no update at all.

In [ ]:
run("""
UPDATE employees_scratch
SET salary = (
    SELECT u.salary
    FROM employee_updates u
    WHERE u.emp_id = employees_scratch.emp_id
    ORDER BY u.updated_at DESC
    LIMIT 1
)
WHERE EXISTS (SELECT 1 FROM employee_updates u WHERE u.emp_id = employees_scratch.emp_id)
""")

show("SELECT emp_id, name, salary FROM employees_scratch ORDER BY emp_id")

**Check 1 - the full salary list.** Employees without an update keep their original value; the five who have one
carry the newest amount.

In [ ]:
assert run("SELECT emp_id, salary FROM employees_scratch ORDER BY emp_id") == [
    (1, 165000), (2, 128000), (3, 125000), (4, 95000), (5, 90000),
    (6, 140000), (7, 101000), (8, 87000), (9, 87000), (10, 105000),
    (11, 88000), (12, 90000), (13, 72000), (14, 68000),
]
print("14 salaries, five of them updated")

**Check 2 - count the changed rows.** Joining the copy to the original shows exactly five differences, which
proves no row was overwritten with `NULL`.

In [ ]:
assert run(
    """
    SELECT COUNT(*) FROM employees_scratch e
    JOIN employees o ON o.emp_id = e.emp_id AND o.salary <> e.salary
    """
) == [(5,)]
print("Q7 passed")

**Check 3 - DuckDB's `UPDATE ... FROM`.** The Postgres-style, set-based spelling: join the newest update
per employee in the `FROM` clause. Rows with no match are untouched by definition, so no `EXISTS` guard
is needed.


In [ ]:
run("DROP TABLE IF EXISTS employees_scratch2")
run("CREATE TABLE employees_scratch2 AS SELECT * FROM employees")
run("""
UPDATE employees_scratch2 SET salary = fresh.salary
FROM (
    SELECT u.emp_id, u.salary,
           ROW_NUMBER() OVER (PARTITION BY u.emp_id ORDER BY u.updated_at DESC) AS rn
    FROM employee_updates u
) fresh
WHERE fresh.emp_id = employees_scratch2.emp_id AND fresh.rn = 1
""")

assert run("SELECT emp_id, salary FROM employees_scratch2 ORDER BY emp_id") == [
    (1, 165000), (2, 128000), (3, 125000), (4, 95000), (5, 90000),
    (6, 140000), (7, 101000), (8, 87000), (9, 87000), (10, 105000),
    (11, 88000), (12, 90000), (13, 72000), (14, 68000),
]
print("UPDATE ... FROM updated the same five salaries")


**Approach and pitfalls.** The correlated scalar subquery with `ORDER BY ... LIMIT 1` picks the newest update,
and the `WHERE EXISTS` guards against overwriting rows with `NULL` when no update exists. Forgetting that guard
silently nulls the salary of every employee missing from the source table - one of the most common production
bugs. DuckDB also supports Postgres-style `UPDATE ... FROM`, used in Check 3 below; the correlated form above is
the most portable across engines.

**Follow-ups.**

1. How would you record the previous salary in an audit table as part of the update?
2. What happens if the subquery can match more than one row and you forget `LIMIT 1`?

---
## Part 3 - Running totals and period-over-period growth

Window functions turn "for each row, also show the aggregate so far" into a single pass. These are the queries
behind dashboards: cumulative revenue, growth rates, and share of total.

### Q8. Running total (overall)

For every order in date order, show the cumulative amount across all orders.

**Expected result.** 14 rows. The running total reaches 1020 at the end of January, 2145 at the end of February
and 3790 at the last order on 2023-03-30.

In [ ]:
show("""
SELECT order_id, order_date, amount,
       SUM(amount) OVER (ORDER BY order_date, order_id) AS running_total
FROM orders
ORDER BY order_date, order_id;
""")

**Check 1 - the cumulative series.** `SUM(amount) OVER (ORDER BY order_date, order_id)` is evaluated after
`WHERE` and walks the ordered rows. Adding the unique `order_id` to the `ORDER BY` makes the total advance row by
row instead of lumping same-date orders together.

In [ ]:
assert run(
    """
    SELECT order_id, order_date, amount,
           SUM(amount) OVER (ORDER BY order_date, order_id) AS running_total
    FROM orders ORDER BY order_date, order_id
    """
) == [
    (11, date(2023, 1, 2), 500, 500),
    (1, date(2023, 1, 5), 100, 600),
    (4, date(2023, 1, 20), 300, 900),
    (7, date(2023, 1, 25), 120, 1020),
    (5, date(2023, 2, 5), 250, 1270),
    (2, date(2023, 2, 10), 200, 1470),
    (9, date(2023, 2, 14), 220, 1690),
    (13, date(2023, 2, 18), 275, 1965),
    (8, date(2023, 2, 28), 180, 2145),
    (12, date(2023, 3, 5), 450, 2595),
    (3, date(2023, 3, 15), 150, 2745),
    (6, date(2023, 3, 20), 400, 3145),
    (14, date(2023, 3, 25), 325, 3470),
    (10, date(2023, 3, 30), 320, 3790),
]
print("running total ends at 3790")


**Check 2 - the endpoint equals the plain total.** A cross-check that the final running total is just
`SUM(amount)` over the table.

In [ ]:
assert run("SELECT SUM(amount) FROM orders") == [(3790,)]
print("Q8 passed")

**Approach and pitfalls.** The default window frame for an ordered `SUM` is `RANGE BETWEEN UNBOUNDED PRECEDING
AND CURRENT ROW`, which means rows that share the same `ORDER BY` value all get the *same* running total. Adding
`order_id` (a unique key) to the `ORDER BY` makes the total advance row by row - the behaviour most people
actually want. Beware that the default `RANGE` frame also ignores peers correctly but can be slow; `ROWS BETWEEN
UNBOUNDED PRECEDING AND CURRENT ROW` is explicit and usually faster.

**Follow-ups.**

1. What is the difference between the `ROWS` and `RANGE` frames when two orders share a date?
2. How would you compute a 3-order moving average with a sliding frame?

### Q9. Running total per customer

Show the cumulative completed-order amount per customer, in order date order.

**Expected result.** 12 rows (non-`completed` orders are excluded). Customer 1 runs 100, 300, 450; customer 5
runs 500 then 950. Customer 4 has a single completed order of 320.

In [ ]:
show("""
SELECT customer_id, order_date, amount,
       SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS running_total
FROM orders
WHERE status = 'completed'
ORDER BY customer_id, order_date, order_id;
""")

**Check 1 - the running total restarts per customer.** There are no non-completed rows in the result, and the
cumulative value resets at every new `customer_id` because of `PARTITION BY`.

In [ ]:
assert run(
    """
    SELECT customer_id, order_date, amount,
           SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS running_total
    FROM orders WHERE status = 'completed'
    ORDER BY customer_id, order_date, order_id
    """
) == [
    (1, date(2023, 1, 5), 100, 100),
    (1, date(2023, 2, 10), 200, 300),
    (1, date(2023, 3, 15), 150, 450),
    (2, date(2023, 1, 20), 300, 300),
    (2, date(2023, 3, 20), 400, 700),
    (3, date(2023, 1, 25), 120, 120),
    (3, date(2023, 2, 28), 180, 300),
    (4, date(2023, 3, 30), 320, 320),
    (5, date(2023, 1, 2), 500, 500),
    (5, date(2023, 3, 5), 450, 950),
    (6, date(2023, 2, 18), 275, 275),
]
print("Q9 passed")


**Approach and pitfalls.** `PARTITION BY customer_id` restarts the running total for each customer; the frame
resets with the partition automatically. The subtle part is what you filter *before* the window: `WHERE status =
'completed'` removes cancelled and pending orders from the sequence entirely. If you instead filtered after the
window, the excluded amounts would still be included in the cumulative sum.

**Follow-ups.**

1. How would you add a column that shows each order's share of that customer's completed revenue?
2. What is the fourth order for customer 2, and why does it not appear here?

### Q10. Month-over-month growth

Using completed orders, show monthly revenue, the previous month's revenue and the month-over-month growth
percentage.

**Expected result.** January 1020 (no previous month, so `NULL`), February 655 with -35.78%, March 1320 with
+101.53%.

In [ ]:
show("""
WITH monthly AS (
    SELECT strftime(order_date, '%Y-%m') AS month,
           SUM(amount) AS revenue
    FROM orders
    WHERE status = 'completed'
    GROUP BY month
)
SELECT month,
       revenue,
       LAG(revenue) OVER (ORDER BY month) AS prev_revenue,
       ROUND(
           100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
           / LAG(revenue) OVER (ORDER BY month),
           2
       ) AS mom_pct
FROM monthly
ORDER BY month;
""")

**Check 1 - monthly revenue and growth.** The aggregate runs first (one row per month), then `LAG` looks at the
previous *month*. January has no predecessor, so its previous value and growth are `NULL`.

In [ ]:
assert run(
    """
    WITH monthly AS (
        SELECT strftime(order_date, '%Y-%m') AS month, SUM(amount) AS revenue
        FROM orders WHERE status = 'completed' GROUP BY month
    )
    SELECT month, revenue, LAG(revenue) OVER (ORDER BY month),
           ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
                 / LAG(revenue) OVER (ORDER BY month), 2)
    FROM monthly ORDER BY month
    """
) == [
    ("2023-01", 1020, None, None),
    ("2023-02", 655, 1020, -35.78),
    ("2023-03", 1320, 655, 101.53),
]
print("mom growth: -35.78%, +101.53%")

**Check 2 - the same growth as a ratio.** Expressing the change as `revenue / prev_revenue` avoids the
subtraction and makes the direction explicit: February is 0.642x January, March is 2.015x February.

In [ ]:
assert run(
    """
    WITH monthly AS (
        SELECT strftime(order_date, '%Y-%m') AS month, SUM(amount) AS revenue
        FROM orders WHERE status = 'completed' GROUP BY month
    )
    SELECT month, ROUND(revenue * 1.0 / LAG(revenue) OVER (ORDER BY month), 3)
    FROM monthly ORDER BY month
    """
) == [("2023-01", None), ("2023-02", 0.642), ("2023-03", 2.015)]
print("Q10 passed")


**Approach and pitfalls.** Aggregate first (one row per month), then apply `LAG` - applying `LAG` to raw rows
gives the previous *order*, not the previous month. Guard against division by zero (a month with revenue 0) and
decide how to treat the first period: `NULL` is usually correct, `0` hides the distinction between "no data" and
"no growth". DuckDB's `/` is always true division (`15 / 2` is `7.5`), so plain `100` would work too; engines with
integer division need the explicit `100.0`.

**Follow-ups.**

1. How would you compute a trailing 3-month average revenue with a window frame?
2. How would you fill in months with no orders so the series has no gaps?

### Q11. Percent of total with a window SUM

Show each customer's completed revenue and its share of all completed revenue.

**Expected result.** Six customers. Customer 5 is the largest at 950 (31.72%); the shares sum to 100.00.

In [ ]:
show("""
SELECT customer_id,
       SUM(amount) AS total,
       ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 2) AS pct_of_total
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
ORDER BY customer_id;
""")

**Check 1 - revenue and share per customer.** `SUM(SUM(amount)) OVER ()` is an aggregate inside a window: the
inner `SUM` collapses each customer group and the outer `SUM(...) OVER ()` totals those group values.

In [ ]:
assert run(
    """
    SELECT customer_id, SUM(amount),
           ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 2)
    FROM orders WHERE status = 'completed'
    GROUP BY customer_id ORDER BY customer_id
    """
) == [
    (1, 450, 15.03),
    (2, 700, 23.37),
    (3, 300, 10.02),
    (4, 320, 10.68),
    (5, 950, 31.72),
    (6, 275, 9.18),
]
print("six customers, customer 5 largest at 31.72%")

**Check 2 - the rounded shares add up to 100.** A sanity check on the denominator: this passes here, but
independent rounding is not guaranteed to sum to exactly 100.00 in general.

In [ ]:
assert run(
    """
    SELECT ROUND(SUM(pct), 2) FROM (
        SELECT ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 2) AS pct
        FROM orders WHERE status = 'completed' GROUP BY customer_id
    )
    """
) == [(100.0,)]
print("Q11 passed")

**Approach and pitfalls.** `SUM(SUM(amount)) OVER ()` is an aggregate *inside* a window: the inner `SUM` collapses
each customer group, and the outer `SUM(...) OVER ()` totals those group values across the whole result. This is
why the query can compute a grand total without a self-join or a second pass. Watch out for integer division
(multiply by `100.0`), and remember rounded percentages may not sum to exactly 100 - the sanity check passes here
but is not guaranteed in general.

**Follow-ups.**

1. How would you compute each customer's share *within its region* instead of globally?
2. How would you rewrite this without window functions?

---
## Part 4 - Joins and set logic

Self-joins, anti-joins and `LEFT JOIN` + `IS NULL` are the bread and butter of dimensional queries. The question
almost always hides a `NULL` or an `EXISTS`/`IN` decision.

### Q12. Self-join for manager and employee

List every employee together with their manager's name. Employees without a manager must still appear.

**Expected result.** 14 rows. Alice, Frank, Judy and Leo show `NULL` as manager; the rest report their manager's
name.

In [ ]:
show("""
SELECT e.emp_id, e.name AS employee, m.name AS manager
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.emp_id
ORDER BY e.emp_id;
""")

**Check 1 - the full roster survives.** A self-join needs two aliases for the same table, and `LEFT JOIN` keeps
the four top-of-hierarchy rows whose `manager_id` is `NULL`. An inner join would silently drop them.

In [ ]:
assert run(
    """
    SELECT e.emp_id, e.name, m.name
    FROM employees e LEFT JOIN employees m ON e.manager_id = m.emp_id
    ORDER BY e.emp_id
    """
) == [
    (1, "Alice", None), (2, "Bob", "Alice"), (3, "Carol", "Alice"),
    (4, "Dave", "Bob"), (5, "Eve", "Bob"),
    (6, "Frank", None), (7, "Grace", "Frank"), (8, "Heidi", "Frank"), (9, "Ivan", "Frank"),
    (10, "Judy", None), (11, "Kevin", "Judy"),
    (12, "Leo", None), (13, "Mallory", "Leo"), (14, "Nina", "Leo"),
]
print("14 employees, 4 with no manager")

**Check 2 - the other direction: employees who manage nobody.** Reversing the self-join and filtering
`COUNT(e.emp_id) = 0` lists the individual contributors; `COUNT(e.emp_id)` is used instead of `COUNT(*)` because
the `LEFT JOIN` would otherwise count the all-`NULL` placeholder row.

In [ ]:
assert run(
    """
    SELECT m.emp_id, m.name, COUNT(e.emp_id) AS reports
    FROM employees m LEFT JOIN employees e ON e.manager_id = m.emp_id
    GROUP BY m.emp_id, m.name HAVING COUNT(e.emp_id) = 0 ORDER BY m.emp_id
    """
) == [
    (3, "Carol", 0), (4, "Dave", 0), (5, "Eve", 0),
    (7, "Grace", 0), (8, "Heidi", 0), (9, "Ivan", 0), (11, "Kevin", 0),
    (13, "Mallory", 0), (14, "Nina", 0),
]
print("Q12 passed")

**Approach and pitfalls.** A self-join needs two aliases so the same table appears twice in the `FROM` clause. Use
`LEFT JOIN` when top-of-hierarchy rows must survive; an inner join silently drops the CEO/root, which is a classic
interview trap. To walk more than one level, use a recursive CTE:

```sql
WITH RECURSIVE chain(emp_id, name, level) AS (
    SELECT emp_id, name, 1 FROM employees WHERE manager_id IS NULL
    UNION ALL
    SELECT e.emp_id, e.name, c.level + 1
    FROM employees e JOIN chain c ON e.manager_id = c.emp_id
)
SELECT * FROM chain ORDER BY level, emp_id;
```

**Follow-ups.**

1. How would you find every employee who ultimately reports to Alice?
2. How would you find employees who earn more than their own manager?

### Q13. Anti-join: customers with no orders

Find customers that have never placed an order.

**Expected result.** Two rows: `(7, 'Gamma SA')` and `(8, 'Hotel BV')`.

In [ ]:
show("""
SELECT c.customer_id, c.name
FROM customers c
WHERE NOT EXISTS (
    SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id
)
ORDER BY c.customer_id;
""")

**Check 1 - `NOT EXISTS`.** The anti-join short-circuits on the first match and is `NULL`-safe, which is why it
is the default answer.

In [ ]:
assert run(
    """
    SELECT c.customer_id, c.name FROM customers c
    WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
    ORDER BY c.customer_id
    """
) == [(7, "Gamma SA"), (8, "Hotel BV")]
print("NOT EXISTS finds Gamma SA and Hotel BV")

**Check 2 - the `LEFT JOIN ... IS NULL` equivalent.** It returns the same rows because `orders.customer_id` is
non-nullable here; that assumption is exactly what to check before rewriting the query.

In [ ]:
assert run(
    """
    SELECT c.customer_id, c.name FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    WHERE o.order_id IS NULL
    ORDER BY c.customer_id
    """
) == [(7, "Gamma SA"), (8, "Hotel BV")]
print("Q13 passed")

**Approach and pitfalls.** `NOT EXISTS` is the safest anti-join: it is `NULL`-proof, short-circuits on the first
match, and optimisers handle it well. `LEFT JOIN ... WHERE right_key IS NULL` is equivalent but only when the
right-hand key is non-nullable - if `orders.customer_id` could be `NULL`, matching "no order" and "an order with a
`NULL` customer" become indistinguishable. `NOT IN (SELECT ...)` is the dangerous one: if the subquery returns a
single `NULL`, the whole predicate becomes unknown and you get zero rows.

**Follow-ups.**

1. Why does `NOT IN` return no rows when the subquery contains a `NULL`?
2. How would you find customers with orders but no *completed* orders?

### Q14. LEFT JOIN and NULL handling

For every customer, count their orders and total order amount, including customers with no orders.

**Expected result.** 8 rows. Gamma SA and Hotel BV show `order_count = 0` and `total_amount = 0` thanks to
`COUNT(o.order_id)` and `COALESCE`. The second check shows what `COUNT(*)` would have produced instead: 1 for
those customers, because the `LEFT JOIN` still emits one all-`NULL` row.

In [ ]:
show("""
SELECT c.customer_id, c.name,
       COUNT(o.order_id) AS order_count,
       COALESCE(SUM(o.amount), 0) AS total_amount
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.name
ORDER BY c.customer_id;
""")

**Check 1 - all eight customers.** `COUNT(o.order_id)` skips the `NULL` produced by the outer join, and
`COALESCE(SUM(...), 0)` turns the empty sum into a real zero.

In [ ]:
assert run(
    """
    SELECT c.customer_id, c.name, COUNT(o.order_id), COALESCE(SUM(o.amount), 0)
    FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.name ORDER BY c.customer_id
    """
) == [
    (1, "Acme Corp", 3, 450),
    (2, "Beta LLC", 3, 950),
    (3, "Cyan Inc", 2, 300),
    (4, "Delta Co", 2, 540),
    (5, "Echo Ltd", 2, 950),
    (6, "Foxtrot GmbH", 2, 600),
    (7, "Gamma SA", 0, 0),
    (8, "Hotel BV", 0, 0),
]
print("zero-order customers report 0, not NULL")

**Check 2 - the `COUNT(*)` pitfall.** Same join, but counting rows instead of non-null keys gives 1 for the two
customers with no orders.

In [ ]:
assert run(
    """
    SELECT c.customer_id, COUNT(*)
    FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id HAVING COUNT(*) = 1 ORDER BY c.customer_id
    """
) == [(7, 1), (8, 1)]
print("Q14 passed")

**Approach and pitfalls.** `COUNT(column)` skips `NULL`s, `COUNT(*)` does not - that single distinction is the
difference between 0 and 1 for customers with no orders. Wrap `SUM` in `COALESCE` because `SUM` over zero rows is
`NULL`, and `NULL` breaks downstream arithmetic. Filtering on a left-joined column in the `WHERE` clause
(`WHERE o.status = 'completed'`) silently turns the outer join back into an inner join; push such predicates into
the `ON` clause when you want to keep the unmatched rows.

**Follow-ups.**

1. How would you keep all customers but count only completed orders?
2. Why does `WHERE o.status = 'completed'` remove customers with no orders?

### Q15. EXISTS vs IN

Find customers that have at least one order, first with `EXISTS` and then with `IN`.

**Expected result.** Both queries return the same six customers, 1 through 6.

In [ ]:
show("""
SELECT c.customer_id, c.name
FROM customers c
WHERE EXISTS (
    SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id
)
ORDER BY c.customer_id;
""")

**Check 1 - the `EXISTS` semi-join.** The engine can stop at the first matching order per customer; `expected` is
then reused by the next check so both forms are compared against one literal.

In [ ]:
expected = [
    (1, "Acme Corp"), (2, "Beta LLC"), (3, "Cyan Inc"),
    (4, "Delta Co"), (5, "Echo Ltd"), (6, "Foxtrot GmbH"),
]

assert run(
    """
    SELECT c.customer_id, c.name FROM customers c
    WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
    ORDER BY c.customer_id
    """
) == expected
print("EXISTS returns 6 customers")

**Check 2 - the `IN` form returns the identical set.** `IN` materialises the subquery list; on a unique,
non-nullable key the optimiser may even produce the same plan.

In [ ]:
assert run(
    """
    SELECT customer_id, name FROM customers
    WHERE customer_id IN (SELECT customer_id FROM orders)
    ORDER BY customer_id
    """
) == expected
print("IN returns the same 6 customers")

**Setup for the `NULL` trap.** A two-row table where one `customer_id` is `NULL` is enough to make `NOT IN`
misbehave.

In [ ]:
run("DROP TABLE IF EXISTS tiny_orders")
run("CREATE TABLE tiny_orders (customer_id INTEGER)")
run("INSERT INTO tiny_orders VALUES (1), (NULL)")
print("tiny_orders holds a 1 and a NULL")

**Check 3 - `NOT IN` with a `NULL` returns no rows.** `x <> NULL` is `UNKNOWN`, so *no* customer can satisfy the
predicate, not even the ones who did not order.

In [ ]:
assert run(
    "SELECT customer_id FROM customers WHERE customer_id NOT IN (SELECT customer_id FROM tiny_orders)"
) == []
print("NOT IN silently matched nothing")

**Check 4 - `NOT EXISTS` handles the same data correctly.** It compares row by row, so the `NULL` row simply
matches nobody and the true anti-join (customers 2-8) comes back.

In [ ]:
assert run(
    "SELECT customer_id FROM customers c WHERE NOT EXISTS "
    "(SELECT 1 FROM tiny_orders t WHERE t.customer_id = c.customer_id) ORDER BY customer_id"
) == [(2,), (3,), (4,), (5,), (6,), (7,), (8,)]
print("Q15 passed")

**Approach and pitfalls.** `EXISTS` is a semi-join: the engine can stop at the first matching row, and it is
`NULL`-safe. `IN` materialises (or hashes) the subquery list and is fine for small, non-nullable lists, but
`NOT IN` with a `NULL` in the list returns *no rows at all*, because `x <> NULL` evaluates to `UNKNOWN`. Modern
optimisers often rewrite `IN` and `EXISTS` into the same plan, so readability and `NULL` safety, not speed, are
the deciding factors.

**Follow-ups.**

1. How do the plans for `EXISTS` and `IN` differ in `EXPLAIN QUERY PLAN`?
2. When does `IN` beat `EXISTS` on performance?

---
## Part 5 - Aggregation, pivots and medians

The next questions combine grouping with window functions: comparing rows to their group, reshaping rows into
columns, and computing order statistics that SQL has no built-in `MEDIAN` for.

### Q16. Employees above their department average

Return employees whose salary is strictly above the average salary of their own department.

**Expected result.** Six rows: Alice, Bob and Carol in Engineering (average 115000), Frank in Sales (average
100500), Judy in Marketing (average 96500) and Leo in Support (average 76666.67). Both the correlated-subquery
and window versions agree.

In [ ]:
show("""
SELECT e.name, e.dept_id, e.salary
FROM employees e
WHERE e.salary > (
    SELECT AVG(salary) FROM employees x WHERE x.dept_id = e.dept_id
)
ORDER BY e.dept_id, e.salary DESC, e.name;
""")

**Check 1 - the correlated subquery answer.** The subquery is conceptually re-evaluated for every employee, each
time restricted to that employee's department. The comparison is a *strict* `>`.

In [ ]:
expected = [
    ("Alice", 1, 150000),
    ("Bob", 1, 120000),
    ("Carol", 1, 120000),
    ("Frank", 2, 130000),
    ("Judy", 3, 105000),
    ("Leo", 4, 90000),
]

assert run(
    """
    SELECT e.name, e.dept_id, e.salary FROM employees e
    WHERE e.salary > (SELECT AVG(salary) FROM employees x WHERE x.dept_id = e.dept_id)
    ORDER BY e.dept_id, e.salary DESC, e.name
    """
) == expected
print("correlated subquery finds 6 employees")

**Check 2 - the window version agrees.** `AVG(salary) OVER (PARTITION BY dept_id)` computes each department
average once, and an outer query filters on it because windows cannot appear in the same `WHERE`.

In [ ]:
assert run(
    """
    SELECT name, dept_id, salary FROM (
        SELECT name, dept_id, salary,
               AVG(salary) OVER (PARTITION BY dept_id) AS dept_avg
        FROM employees
    ) WHERE salary > dept_avg
    ORDER BY dept_id, salary DESC, name
    """
) == expected
print("window version matches")

**Check 3 - the averages, for reference.** These are the thresholds every comparison above used.

In [ ]:
assert run(
    """
    SELECT dept_id, ROUND(AVG(salary), 2) FROM employees GROUP BY dept_id ORDER BY dept_id
    """
) == [(1, 115000.0), (2, 100500.0), (3, 96500.0), (4, 76666.67)]
print("Q16 passed")

**Approach and pitfalls.** The correlated subquery is the textbook answer and reads clearly, but it is
conceptually one aggregate per row. The window version computes each department average once per partition and is
usually the faster plan on large tables. Both compare against a *strict* `>`; using `>=` would pull in everyone in
a department where every salary equals the average. Note the window function cannot appear in the same `WHERE`,
which is why the window version needs the extra subquery level.

**Follow-ups.**

1. How would you return each employee's salary as a percentage of their department total?
2. Which version would you choose for a 1-billion-row fact table, and why?

### Q17. Pivot with CASE

Turn the `status` values into columns: one row per customer with the number of completed, pending and cancelled
orders.

**Expected result.** 8 rows. Beta LLC has 2 completed and 1 cancelled; Delta Co has 1 completed and 1 pending; the
two customers with no orders show zeros.

In [ ]:
show("""
SELECT c.customer_id, c.name,
       COUNT(CASE WHEN o.status = 'completed' THEN 1 END) AS completed,
       COUNT(CASE WHEN o.status = 'pending'   THEN 1 END) AS pending,
       COUNT(CASE WHEN o.status = 'cancelled' THEN 1 END) AS cancelled
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.name
ORDER BY c.customer_id;
""")

**Check 1 - pivoting counts.** `COUNT(CASE WHEN ... THEN 1 END)` has no `ELSE`, so unmatched statuses produce
`NULL`, which `COUNT` ignores - that is what yields a clean 0 for customers with no orders.

In [ ]:
assert run(
    """
    SELECT c.customer_id, c.name,
           COUNT(CASE WHEN o.status = 'completed' THEN 1 END),
           COUNT(CASE WHEN o.status = 'pending'   THEN 1 END),
           COUNT(CASE WHEN o.status = 'cancelled' THEN 1 END)
    FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.name ORDER BY c.customer_id
    """
) == [
    (1, "Acme Corp", 3, 0, 0),
    (2, "Beta LLC", 2, 0, 1),
    (3, "Cyan Inc", 2, 0, 0),
    (4, "Delta Co", 1, 1, 0),
    (5, "Echo Ltd", 2, 0, 0),
    (6, "Foxtrot GmbH", 1, 0, 1),
    (7, "Gamma SA", 0, 0, 0),
    (8, "Hotel BV", 0, 0, 0),
]
print("count pivot: 8 customers, 3 status columns")

**Check 2 - pivoting amounts instead of counts.** With `SUM` the `ELSE 0` is mandatory: without it the whole sum
for a customer with no matching rows would be `NULL`.

In [ ]:
assert run(
    """
    SELECT c.customer_id,
           SUM(CASE WHEN o.status = 'completed' THEN o.amount ELSE 0 END),
           SUM(CASE WHEN o.status = 'pending'   THEN o.amount ELSE 0 END),
           SUM(CASE WHEN o.status = 'cancelled' THEN o.amount ELSE 0 END)
    FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id ORDER BY c.customer_id
    """
) == [
    (1, 450, 0, 0),
    (2, 700, 0, 250),
    (3, 300, 0, 0),
    (4, 320, 220, 0),
    (5, 950, 0, 0),
    (6, 275, 0, 325),
    (7, 0, 0, 0),
    (8, 0, 0, 0),
]
print("Q17 passed")

**Real `PIVOT` too.** DuckDB has an actual `PIVOT` clause, so the column set can come from the data
instead of hand-written `CASE`s. Pivot columns are ordered alphabetically by default (`cancelled`,
`completed`, `pending`), and `count(*)` contributes `0` for combinations that do not occur.


In [ ]:
show("""
SELECT * FROM (
    PIVOT orders ON status USING count(*) GROUP BY customer_id
) ORDER BY customer_id;
""")

assert run("""
SELECT customer_id, cancelled, completed, pending FROM (
    PIVOT orders ON status USING count(*) GROUP BY customer_id
) ORDER BY customer_id
""") == [
    (1, 0, 3, 0),
    (2, 1, 2, 0),
    (3, 0, 2, 0),
    (4, 0, 1, 1),
    (5, 0, 2, 0),
    (6, 1, 1, 0),
]
print("PIVOT agrees with the CASE pivot")


**Check - the round trip.** `UNPIVOT` is the declared reverse: columns back into rows. It drops `NULL`
values by default while zeros survive as real rows, so this `COUNT`-pivoted result reproduces the plain
status counts exactly.


In [ ]:
assert run("""
WITH pivoted AS (
    PIVOT orders ON status USING count(*) GROUP BY customer_id
)
SELECT status, SUM(orders_count) AS n_orders FROM (
    UNPIVOT pivoted ON cancelled, completed, pending INTO NAME status VALUE orders_count
) GROUP BY status ORDER BY status
""") == [("cancelled", 2), ("completed", 11), ("pending", 1)]
print("UNPIVOT round-trips to the raw status counts")


**Approach and pitfalls.** For counts, `COUNT(CASE WHEN ... THEN 1 END)` is the safe form: the missing `ELSE`
yields `NULL`, which `COUNT` ignores, so unmatched statuses contribute 0. With `SUM` you need `ELSE 0`,
otherwise the whole sum is `NULL` for customers with no matching rows. DuckDB also offers `COUNT(*) FILTER
(WHERE o.status = 'completed')` - a cleaner conditional-aggregation spelling - and, unlike most engines, a
real `PIVOT`/`UNPIVOT` clause for the dynamic case (the two checks above).


**Follow-ups.**

1. Rewrite the count pivot using `FILTER (WHERE ...)`.
2. How would you produce the same pivot dynamically when the status values are unknown?

### Q18. Median salary

Compute the median of `employees.salary` - with the engine's own `median()` and with the portable window
technique, so you can speak to both.

**Expected result.** All checks return `92500.0`. With 14 salaries the median is the average of the 7th and 8th
values (90000 and 95000); DuckDB interpolates, so the result arrives as a float.


In [ ]:
show("""
SELECT median(salary) AS median_salary,
       quantile_cont(salary, 0.5) AS p50
FROM employees;
""")


**Check 1 - the engine-native answer.** `median()` and `quantile_cont(salary, 0.5)` agree in one line; both
interpolate, so even and odd row counts need no manual middle-row handling.


In [ ]:
assert run("SELECT median(salary), quantile_cont(salary, 0.5) FROM employees") == [(92500.0, 92500.0)]
print("median via median()/quantile_cont:", 92500.0)


**Check 2 - the portable window version.** Rank the rows and average the middle two; for odd `n` the two
computed ranks land on the same row. Division note: DuckDB's `/` is true division (`15 / 2` is `7.5`), so
integer rank arithmetic needs `//` - the opposite convention of engines where `/` truncates.


In [ ]:
assert run(
    """
    WITH ranked AS (
        SELECT salary, ROW_NUMBER() OVER (ORDER BY salary) AS rn, COUNT(*) OVER () AS n
        FROM employees
    )
    SELECT AVG(salary) FROM ranked WHERE rn IN ((n + 1) // 2, (n + 2) // 2)
    """
) == [(92500.0,)]
print("median via ROW_NUMBER:", 92500.0)


**Check 3 - the two middle values.** Confirms that the median is the average of 90000 and 95000 rather than one
of them.

In [ ]:
assert run(
    """
    SELECT salary FROM (
        SELECT salary, ROW_NUMBER() OVER (ORDER BY salary) AS rn, COUNT(*) OVER () AS n
        FROM employees
    ) WHERE rn IN ((n + 1) // 2, (n + 2) // 2) ORDER BY salary
    """
) == [(90000,), (95000,)]
print("Q18 passed")

**Approach and pitfalls.** `median()` is the one-liner and interpolates exactly like `PERCENTILE_CONT` in
BigQuery/Snowflake/Postgres; `quantile_cont(salary, 0.9)` is the 90th percentile. The portable window version
is what you reach for when the engine has no native function - watch `/` vs `//` in DuckDB, and remember that
even counts produce the mean of the two middle rows as a float. `LIMIT`/`OFFSET` tricks also work but are
harder to read; prefer the functions.


**Follow-ups.**

1. How would you compute the median per department?
2. How would you compute the 90th percentile salary?

---
## Part 6 - Time, sequences and cohorts

These are the hardest questions to improvise: gaps and islands, sessionization, and cohort retention all reduce to
"compare each row with the previous row in order" - which is `LAG` plus a cumulative `SUM`.

### Q19. Consecutive-day streak (gaps and islands)

For each user in `events`, find their longest run of consecutive calendar days with at least one event.

**Expected result.** User 1 has a 3-day streak (Jan 1-3) plus a 2-day streak (Jan 5-6); user 2 has a 2-day streak;
users 3 and 4 each have a single day.

In [ ]:
show("""
WITH daily AS (
    SELECT DISTINCT user_id, CAST(event_ts AS DATE) AS d
    FROM events
),
grouped AS (
    SELECT user_id, d,
           d - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY d)::INTEGER AS island
    FROM daily
)
SELECT user_id, MIN(d) AS start_day, MAX(d) AS end_day, COUNT(*) AS streak_len
FROM grouped
GROUP BY user_id, island
ORDER BY user_id, start_day;
""")


**Check 1 - every island.** Deduplicating to one row per day *first* keeps `date - row_number` constant inside a
run, so grouping by that constant recovers the islands.

In [ ]:
assert run(
    """
    WITH daily AS (
        SELECT DISTINCT user_id, CAST(event_ts AS DATE) AS d FROM events
    ), grouped AS (
        SELECT user_id, d,
               d - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY d)::INTEGER AS island
        FROM daily
    )
    SELECT user_id, MIN(d) AS start_day, MAX(d), COUNT(*)
    FROM grouped GROUP BY user_id, island ORDER BY user_id, start_day
    """
) == [
    (1, date(2023, 1, 1), date(2023, 1, 3), 3),
    (1, date(2023, 1, 5), date(2023, 1, 6), 2),
    (2, date(2023, 1, 1), date(2023, 1, 2), 2),
    (2, date(2023, 1, 4), date(2023, 1, 4), 1),
    (3, date(2023, 2, 1), date(2023, 2, 1), 1),
    (4, date(2023, 2, 1), date(2023, 2, 1), 1),
]
print("6 islands across 4 users")


**Check 2 - the longest streak per user.** Wrapping the island query and taking `MAX(streak_len)` gives the
one-number answer, 3 for user 1 and 2 for user 2.

In [ ]:
assert run(
    """
    WITH daily AS (
        SELECT DISTINCT user_id, CAST(event_ts AS DATE) AS d FROM events
    ), grouped AS (
        SELECT user_id, d,
               d - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY d)::INTEGER AS island
        FROM daily
    )
    SELECT user_id, MAX(streak_len) FROM (
        SELECT user_id, island, COUNT(*) AS streak_len FROM grouped GROUP BY user_id, island
    ) GROUP BY user_id ORDER BY user_id
    """
) == [(1, 3), (2, 2), (3, 1), (4, 1)]
print("Q19 passed")


**Approach and pitfalls.** The gaps-and-islands trick: within a consecutive run, `date - row_number` is constant,
because both advance by one each day. Group by that constant "island key" and you have your runs. Deduplicate to
one row per day *first* (`SELECT DISTINCT`), or multiple events on the same day inflate the count and shift the
row numbers. Here the trick is `d - rn`: DuckDB subtracts an integer number of
days from a `DATE` (the `::INTEGER` cast matters, because `ROW_NUMBER` returns a `BIGINT` and `DATE -
BIGINT` has no overload). Postgres uses `d - n` too; BigQuery has `DATE_SUB(d, INTERVAL n DAY)`. Streaks across month or year
boundaries work automatically because real dates are used, not string parts.

**Follow-ups.**

1. How would you adapt this to count consecutive *orders* (by order id) rather than calendar days?
2. How would you list every user whose longest streak is at least 3?

### Q20. Sessionization with LAG

Assign a session id to each event: a new session starts when the gap from the same user's previous event exceeds
30 minutes.

**Expected result.** User 3 gets two sessions - events at 10:00, 10:05 and 10:25 are one session, then 11:30
starts a second (the 65-minute gap) and 11:45 joins it. User 4's two events are 20 minutes apart, so one session.
Across all users the session counts are 5, 3, 2 and 1.

In [ ]:
show("""
WITH ordered AS (
    SELECT event_id, user_id, event_ts, event_type,
           LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS prev_ts
    FROM events
),
flagged AS (
    SELECT event_id, user_id, event_ts, event_type,
           CASE WHEN prev_ts IS NULL
                  OR date_diff('second', prev_ts, event_ts) / 60.0 > 30
                THEN 1 ELSE 0 END AS is_new_session
    FROM ordered
)
SELECT user_id, event_ts, event_type,
       SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS session_id
FROM flagged
WHERE user_id IN (3, 4)
ORDER BY user_id, event_ts;
""")


**Check 1 - the session ids for users 3 and 4.** `LAG` finds the previous event, the `CASE` flags a boundary
(including the first event, whose `prev_ts` is `NULL`), and the cumulative `SUM` numbers the sessions per user.

In [ ]:
assert run(
    """
    WITH ordered AS (
        SELECT event_id, user_id, event_ts, event_type,
               LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS prev_ts
        FROM events
    ), flagged AS (
        SELECT event_id, user_id, event_ts, event_type,
               CASE WHEN prev_ts IS NULL
                      OR date_diff('second', prev_ts, event_ts) / 60.0 > 30
                    THEN 1 ELSE 0 END AS is_new_session
        FROM ordered
    )
    SELECT user_id, event_ts, event_type,
           SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_ts, event_id)
    FROM flagged WHERE user_id IN (3, 4) ORDER BY user_id, event_ts
    """
) == [
    (3, datetime(2023, 2, 1, 10, 0), "view", 1),
    (3, datetime(2023, 2, 1, 10, 5), "click", 1),
    (3, datetime(2023, 2, 1, 10, 25), "view", 1),
    (3, datetime(2023, 2, 1, 11, 30), "purchase", 2),
    (3, datetime(2023, 2, 1, 11, 45), "view", 2),
    (4, datetime(2023, 2, 1, 10, 0), "login", 1),
    (4, datetime(2023, 2, 1, 10, 20), "view", 1),
]
print("user 3 has 2 sessions, user 4 has 1")


**Check 2 - session count per user.** The largest session id per user *is* the number of sessions; grouping by
`user_id` and taking `MAX(session_id)` returns 5, 3, 2 and 1.

In [ ]:
assert run(
    """
    WITH ordered AS (
        SELECT event_id, user_id, event_ts,
               LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS prev_ts
        FROM events
    ), flagged AS (
        SELECT event_id, user_id, event_ts,
               CASE WHEN prev_ts IS NULL
                      OR date_diff('second', prev_ts, event_ts) / 60.0 > 30
                    THEN 1 ELSE 0 END AS is_new_session
        FROM ordered
    ), numbered AS (
        SELECT user_id,
               SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS session_id
        FROM flagged
    )
    SELECT user_id, MAX(session_id) FROM numbered GROUP BY user_id ORDER BY user_id
    """
) == [(1, 5), (2, 3), (3, 2), (4, 1)]
print("Q20 passed")

**Approach and pitfalls.** Sessionization is "flag the boundaries, then cumulative-sum the flags". `LAG` needs a
deterministic `ORDER BY` (tie-break with `event_id`) or events in the same second can be reordered. `date_diff('second', ...)` counts second
boundaries crossed, so a 65-minute gap is `3900` seconds and `/ 60.0` is a float minutes value; the minute
variant would truncate sub-minute gaps. The first event per user has `prev_ts IS NULL` and must be flagged as a new session
explicitly. Session ids are per user and start at 1 because the `SUM` is partitioned.

**Follow-ups.**

1. How would you also start a new session when the event type changes (for example a `purchase` closes a session)?
2. How would you materialise a `sessions` table with start time, end time and event count?

### Q21. Cohort and retention by month

Group customers into cohorts by the month of their first order, then count how many customers from each cohort
were active in each subsequent month (including the acquisition month).

**Expected result.** The January 2023 cohort has 4 customers and retains 4, 3, 3 across January, February and
March. The February cohort has 2 customers, retained 2 and 2.

In [ ]:
show("""
WITH first_order AS (
    SELECT customer_id, MIN(strftime(order_date, '%Y-%m')) AS cohort_month
    FROM orders
    GROUP BY customer_id
),
activity AS (
    SELECT DISTINCT customer_id, strftime(order_date, '%Y-%m') AS activity_month
    FROM orders
)
SELECT f.cohort_month,
       a.activity_month,
       COUNT(DISTINCT a.customer_id) AS active_customers
FROM first_order f
JOIN activity a ON a.customer_id = f.customer_id
GROUP BY f.cohort_month, a.activity_month
ORDER BY f.cohort_month, a.activity_month;
""")

**Check 1 - the retention triangle.** Three steps: find each customer's first month, list the distinct
(customer, month) activity pairs, then join and count. `COUNT(DISTINCT customer_id)` stops a customer with
several orders in one month from inflating the count.

In [ ]:
assert run(
    """
    WITH first_order AS (
        SELECT customer_id, MIN(strftime(order_date, '%Y-%m')) AS cohort_month FROM orders GROUP BY customer_id
    ), activity AS (
        SELECT DISTINCT customer_id, strftime(order_date, '%Y-%m') AS activity_month FROM orders
    )
    SELECT f.cohort_month, a.activity_month, COUNT(DISTINCT a.customer_id)
    FROM first_order f JOIN activity a ON a.customer_id = f.customer_id
    GROUP BY f.cohort_month, a.activity_month ORDER BY f.cohort_month, a.activity_month
    """
) == [
    ("2023-01", "2023-01", 4),
    ("2023-01", "2023-02", 3),
    ("2023-01", "2023-03", 3),
    ("2023-02", "2023-02", 2),
    ("2023-02", "2023-03", 2),
]
print("January cohort retains 4, 3, 3")

**Check 2 - cohort sizes.** The denominator for any retention rate: 4 customers acquired in January and 2 in
February.

In [ ]:
assert run(
    """
    WITH first_order AS (
        SELECT customer_id, MIN(strftime(order_date, '%Y-%m')) AS cohort_month FROM orders GROUP BY customer_id
    )
    SELECT cohort_month, COUNT(*) FROM first_order GROUP BY cohort_month ORDER BY cohort_month
    """
) == [("2023-01", 4), ("2023-02", 2)]
print("Q21 passed")

**Approach and pitfalls.** Three steps: (1) one row per customer with the first period (the cohort), (2) the
distinct (customer, period) pairs that were active, (3) join and count. `COUNT(DISTINCT customer_id)` is the
safety net - without it a customer with three orders in one month inflates retention above 100%. The month offset
is usually more useful than the raw month label:

```sql
date_diff('month', strptime(f.cohort_month, '%Y-%m')::DATE,
                   strptime(a.activity_month, '%Y-%m')::DATE)
```

Also decide whether "active" means any order or only completed orders; here it is any order.

**Follow-ups.**

1. Turn the counts into a retention rate using the cohort sizes.
2. How would you pivot the result into a classic cohort matrix (one row per cohort, one column per month offset)?

### Q22. Cumulative distinct count

For each order date, show how many distinct customers had placed an order on or before that date.

**Expected result.** 14 rows. The count reaches 4 by 2023-01-25, 5 on 2023-02-14 and 6 on 2023-02-18, then stays
at 6 - all six ordering customers have been seen.

In [ ]:
show("""
SELECT o.order_date,
       (
           SELECT COUNT(DISTINCT o2.customer_id)
           FROM orders o2
           WHERE o2.order_date <= o.order_date
       ) AS cum_distinct_customers
FROM (SELECT DISTINCT order_date FROM orders) o
ORDER BY o.order_date;
""")

**Check 1 - the correlated-subquery answer.** Correct but O(dates x rows): for each distinct date, count the
distinct customers that had ordered by then.

In [ ]:
assert run(
    """
    SELECT o.order_date,
           (SELECT COUNT(DISTINCT o2.customer_id) FROM orders o2 WHERE o2.order_date <= o.order_date)
    FROM (SELECT DISTINCT order_date FROM orders) o ORDER BY o.order_date
    """
) == [
    (date(2023, 1, 2), 1),
    (date(2023, 1, 5), 2),
    (date(2023, 1, 20), 3),
    (date(2023, 1, 25), 4),
    (date(2023, 2, 5), 4),
    (date(2023, 2, 10), 4),
    (date(2023, 2, 14), 5),
    (date(2023, 2, 18), 6),
    (date(2023, 2, 28), 6),
    (date(2023, 3, 5), 6),
    (date(2023, 3, 15), 6),
    (date(2023, 3, 20), 6),
    (date(2023, 3, 25), 6),
    (date(2023, 3, 30), 6),
]
print("cumulative distinct reaches 6")


**Check 2 - the scalable window alternative.** Instead of recounting history, find each customer's first order
date and cumulate the count of *new* customers with `SUM(COUNT(*)) OVER (ORDER BY first_date)`.

In [ ]:
assert run(
    """
    WITH first_order AS (
        SELECT customer_id, MIN(order_date) AS first_date FROM orders GROUP BY customer_id
    )
    SELECT first_date, COUNT(*), SUM(COUNT(*)) OVER (ORDER BY first_date)
    FROM first_order GROUP BY first_date ORDER BY first_date
    """
) == [
    (date(2023, 1, 2), 1, 1),
    (date(2023, 1, 5), 1, 2),
    (date(2023, 1, 20), 1, 3),
    (date(2023, 1, 25), 1, 4),
    (date(2023, 2, 14), 1, 5),
    (date(2023, 2, 18), 1, 6),
]
print("Q22 passed")


**Check 3 - DuckDB's one-shot window.** DuckDB does support `COUNT(DISTINCT ...) OVER (ORDER BY ...)`, so
the cumulative distinct count can be written directly over the distinct (date, customer) pairs. Mind the
frames: the default `RANGE` frame is what makes same-date peers correct here.


In [ ]:
assert run(
    """
    WITH seen AS (SELECT DISTINCT order_date, customer_id FROM orders)
    SELECT order_date, COUNT(DISTINCT customer_id) OVER (ORDER BY order_date) AS cum_distinct_customers
    FROM seen ORDER BY order_date
    """
) == [
    (date(2023, 1, 2), 1), (date(2023, 1, 5), 2), (date(2023, 1, 20), 3), (date(2023, 1, 25), 4),
    (date(2023, 2, 5), 4), (date(2023, 2, 10), 4), (date(2023, 2, 14), 5), (date(2023, 2, 18), 6),
    (date(2023, 2, 28), 6), (date(2023, 3, 5), 6), (date(2023, 3, 15), 6), (date(2023, 3, 20), 6),
    (date(2023, 3, 25), 6), (date(2023, 3, 30), 6),
]
print("COUNT(DISTINCT) OVER works in DuckDB")


**Approach and pitfalls.** There are three spellings for a cumulative distinct count. The correlated subquery
(Check 1) is correct but O(dates x rows). DuckDB directly supports `COUNT(DISTINCT ...) OVER (ORDER BY date)`
(Check 3), but its frame semantics deserve care and it recounts the history for every row. The one that scales
is Check 2: find each customer's first order date, and cumulate the count of *new* customers with
`SUM(COUNT(*)) OVER (ORDER BY first_date)` - it naturally answers "how many customers had we acquired by date X".
A common wrong answer is `SUM(COUNT(DISTINCT customer_id))` over raw periods, which double-counts customers who
order in several periods.


**Follow-ups.**

1. How would you produce a daily active-user count alongside the cumulative distinct count?
2. How would you compute a 7-day rolling distinct customer count?

---
## Part 7 - DuckDB skills that SQLite could not teach

DuckDB is a columnar, vectorised analytics engine, so it ships SQL that an embedded row-store cannot:
window filtering in one clause, real `PIVOT`/`UNPIVOT`, point-in-time joins with `ASOF`, typed nested
values, and plans you can actually read. Every lesson below runs on the seed tables with the same
`show()` + `assert` discipline as the question bank - muscle memory, not illustration.


### Skill 1 - `QUALIFY`: filtering on a window result

Parts 1-4 repeated one mantra: "filter on the ranking in an outer query, because window functions are
evaluated after `WHERE`". DuckDB adds `QUALIFY` for exactly that: it is to window functions what `HAVING`
is to aggregates, and it needs one clause instead of a subquery.


In [ ]:
show("""
SELECT emp_id, salary, updated_at
FROM employee_updates
QUALIFY ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY updated_at DESC, rowid DESC) = 1
ORDER BY emp_id;
""")


**Check 1 - the same five rows as Q4, minus the subquery.** The window function lives at the bottom of the
same `SELECT`; no derived table, no outer filter.


In [ ]:
assert run("""
SELECT emp_id, dept_id, salary, updated_at FROM employee_updates
QUALIFY ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY updated_at DESC, rowid DESC) = 1
ORDER BY emp_id
""") == [
    (1, 1, 165000, date(2023, 6, 1)),
    (2, 1, 128000, date(2023, 5, 1)),
    (3, 1, 125000, date(2023, 3, 1)),
    (6, 2, 140000, date(2023, 7, 1)),
    (7, 2, 101000, date(2023, 4, 1)),
]
print("Q4 via a single QUALIFY")


**Check 2 - rewrite Q3's top-1-per-group in one clause.** `PARTITION BY` works as before; only the filter
moves up into `QUALIFY`.


In [ ]:
assert run("""
SELECT dept_id, name, salary FROM employees
QUALIFY ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC, emp_id) = 1
ORDER BY dept_id
""") == [
    (1, "Alice", 150000),
    (2, "Frank", 130000),
    (3, "Judy", 105000),
    (4, "Leo", 90000),
]
print("top earner per department via QUALIFY")


**Approach and pitfalls.** `QUALIFY` is a vendor extension - DuckDB, Snowflake and BigQuery support it;
Postgres and MySQL do not, so keep the subquery form ready for portability (and SQLite answers either way
with the subquery). Evaluation order is `FROM` -> `WHERE` -> `GROUP BY`/`HAVING` -> window functions ->
`QUALIFY` -> `ORDER BY`/`LIMIT`, which is why `QUALIFY` can see window aliases that `WHERE` cannot.

**Follow-ups.**

1. How would you express Q16 ("above department average") with `QUALIFY`?
2. For a 1-billion-row table, when would `QUALIFY` beat a pre-aggregated subquery?


### Skill 2 - `PIVOT`/`UNPIVOT`: real reshaping syntax

Q17 conditioned on known statuses with hand-written `CASE`s. DuckDB's `PIVOT` derives the columns from the
data itself, and `UNPIVOT` is its declared reverse.


In [ ]:
show("""
SELECT * FROM (
    PIVOT orders ON status USING count(*) GROUP BY customer_id
) ORDER BY customer_id;
""")


**Check 1 - the derived pivot.** Columns come from the `status` values (alphabetical by default), and a
combination with no rows counts `0`.


In [ ]:
assert run("""
SELECT customer_id, cancelled, completed, pending FROM (
    PIVOT orders ON status USING count(*) GROUP BY customer_id
) ORDER BY customer_id
""") == [
    (1, 0, 3, 0),
    (2, 1, 2, 0),
    (3, 0, 2, 0),
    (4, 0, 1, 1),
    (5, 0, 2, 0),
    (6, 1, 1, 0),
]
print("PIVOT derives the status columns")


**Check 2 - the round trip.** `UNPIVOT` puts the columns back into rows. It drops `NULL` values by default
while zeros survive as real rows, so this `COUNT`-pivoted result reproduces the plain status counts
exactly.


In [ ]:
assert run("""
WITH pivoted AS (
    PIVOT orders ON status USING count(*) GROUP BY customer_id
)
SELECT status, SUM(orders_count) AS n_orders FROM (
    UNPIVOT pivoted ON cancelled, completed, pending INTO NAME status VALUE orders_count
) GROUP BY status ORDER BY status
""") == [("cancelled", 2), ("completed", 11), ("pending", 1)]
print("UNPIVOT round-trips to the raw status counts")


**Approach and pitfalls.** Standard SQL has no `PIVOT`; conditional aggregation (Q17's `CASE` form) is the
portable spelling and what most engines expect. DuckDB's clause-based form is dynamic - add a fourth
status and a new column appears - which is exactly Q17's follow-up question answered. Two sharp edges:
pivot columns are ordered alphabetically unless the values are listed (`ON status IN ('completed',
'pending', 'cancelled')`), and `UNPIVOT` silently drops rows whose value is `NULL`.


### Skill 3 - `ASOF JOIN`: point-in-time lookup

"Which rate was in effect when the order happened?" - the latest row at or before a timestamp. The usual
answers are a windowed deduplication (Q4) or a correlated `ORDER BY ... LIMIT 1` (Q7). DuckDB has a
first-class join for it, so this lesson adds one seed table: FX rates published on specific dates.


In [ ]:
run("DROP TABLE IF EXISTS fx_rates")
run("""
CREATE TABLE fx_rates (
    effective_date DATE NOT NULL,
    usd_eur        DOUBLE NOT NULL
);
""")

run("""
INSERT INTO fx_rates (effective_date, usd_eur) VALUES
    ('2023-01-02', 1.05),
    ('2023-01-09', 1.08),
    ('2023-01-16', 1.10),
    ('2023-02-06', 1.09),
    ('2023-02-20', 1.11),
    ('2023-03-06', 1.12),
    ('2023-03-20', 1.13);
""")

show("SELECT * FROM fx_rates ORDER BY effective_date")


Every order converts to EUR at the most recently *published* rate: `ASOF JOIN` picks, for each left row,
the right row with the greatest matching timestamp that is not after it.


In [ ]:
show("""
SELECT o.order_id, o.order_date, f.effective_date AS rate_from, f.usd_eur
FROM orders o
ASOF JOIN fx_rates f ON o.order_date >= f.effective_date
ORDER BY o.order_id;
""")

assert run("""
SELECT o.order_id, f.usd_eur
FROM orders o ASOF JOIN fx_rates f ON o.order_date >= f.effective_date
ORDER BY o.order_id
""") == [
    (1, 1.05), (2, 1.09), (3, 1.12), (4, 1.10), (5, 1.10), (6, 1.13), (7, 1.10),
    (8, 1.11), (9, 1.09), (10, 1.13), (11, 1.05), (12, 1.11), (13, 1.09), (14, 1.13),
]
print("per-order rate is always the latest published one")


**Check 2 - and the conversion computes.** Completed orders times the rate in effect; the total is exactly
the per-order mapping above.


In [ ]:
assert run("""
SELECT ROUND(SUM(amount * usd_eur), 2)
FROM orders o ASOF JOIN fx_rates f ON o.order_date >= f.effective_date
WHERE o.status = 'completed'
""") == [(3290.65,)]
print("completed revenue in EUR: 3290.65")


**Approach and pitfalls.** `ASOF JOIN` needs at least one inequality (`>=` here) plus any plain equality
conditions. Pitfalls: the compared columns must be the same type type (a `DATE` against a `TIMESTAMP`
compares, but silently widen - keep both `DATE`s when the grain is days); duplicate effective timestamps
make the pick non-deterministic, so keep one row per effective date; and the table has no `WHERE` - filter
the right side in the join condition. The portable spelling is the correlated `ORDER BY ... LIMIT 1` from
Q7; Spark has `joinAsof`, Snowflake `ASOF JOIN`, Postgres requires the lateral version.


### Skill 4 - `LIST` and `STRUCT`: typed nested values

DuckDB can aggregate into nested, *typed* containers: a `LIST` of `STRUCT`s is a real column type, and it
arrives in Python as a native list of dicts, not a JSON string.


In [ ]:
show("""
SELECT dept_id,
       list(struct_pack(name := name, salary := salary) ORDER BY salary DESC, emp_id) AS roster
FROM employees
GROUP BY dept_id
ORDER BY dept_id;
""")


**Check 1 - the nested values are typed, not stringified.** Each department aggregates to a `LIST` of
`STRUCT`s, ordered by the `ORDER BY` inside the aggregate.


In [ ]:
assert run("""
SELECT dept_id, list(struct_pack(name := name, salary := salary) ORDER BY salary DESC, emp_id)
FROM employees GROUP BY dept_id ORDER BY dept_id
""") == [
    (1, [{"name": "Alice", "salary": 150000}, {"name": "Bob", "salary": 120000},
         {"name": "Carol", "salary": 120000}, {"name": "Dave", "salary": 95000},
         {"name": "Eve", "salary": 90000}]),
    (2, [{"name": "Frank", "salary": 130000}, {"name": "Grace", "salary": 98000},
         {"name": "Heidi", "salary": 87000}, {"name": "Ivan", "salary": 87000}]),
    (3, [{"name": "Judy", "salary": 105000}, {"name": "Kevin", "salary": 88000}]),
    (4, [{"name": "Leo", "salary": 90000}, {"name": "Mallory", "salary": 72000},
         {"name": "Nina", "salary": 68000}]),
]
print("LIST<STRUCT> rosters are native Python lists of dicts")


**Check 2 - query *into* the nested values.** List indexing is 1-based (off by one from Python lists) and
struct fields are reached with `.field`.


In [ ]:
assert run("""
SELECT dept_id, roster[1].name AS top_earner
FROM (
    SELECT dept_id, list(struct_pack(name := name, salary := salary) ORDER BY salary DESC, emp_id) AS roster
    FROM employees GROUP BY dept_id
) ORDER BY dept_id
""") == [(1, "Alice"), (2, "Frank"), (3, "Judy"), (4, "Leo")]
print("1-based list index, dotted struct access")


**Check 3 - a `STRUCT` of aggregates.** A whole group's summary folds into one struct value - the shape a
semi-structured payload or a nested fact table takes.


In [ ]:
assert run("""
SELECT dept_id, struct_pack(headcount := COUNT(*), payroll := SUM(salary))
FROM employees GROUP BY dept_id ORDER BY dept_id
""") == [
    (1, {"headcount": 5, "payroll": 575000}),
    (2, {"headcount": 4, "payroll": 402000}),
    (3, {"headcount": 2, "payroll": 193000}),
    (4, {"headcount": 3, "payroll": 230000}),
]
print("STRUCT of aggregates per department")


**Approach and pitfalls.** Nested types shine for top-N-per-group without joins and for carrying
semi-structured payloads through a pipeline. Syntax worth knowing: `struct_pack(name := x, ...)` builds a
struct, `list(x ORDER BY y)` aggregates into a list, `roster[1]` and `st.field` read back, `UNNEST`
explodes a list into rows, and `len(l)` is the list length. Indexing is 1-based. For open-ended keys there
is a native `MAP` type.

**Follow-ups.**

1. How would you explode the rosters back into one row per employee with `UNNEST`?
2. When is a `MAP` better than a `STRUCT` for a payload?


### Skill 5 - reading `EXPLAIN` and `EXPLAIN ANALYZE`

T2's card introduced the vocabulary; here is the full walkthrough. DuckDB prints the physical plan as an
ASCII tree of operators, read top-down: output at the top, storage at the bottom.


In [ ]:
plan_text = run("""
EXPLAIN SELECT e.name, d.dept_name
FROM employees e JOIN departments d ON e.dept_id = d.dept_id
""")[0][1]
print(plan_text)

assert plan_text.count("SEQ_SCAN") >= 2
join_ops = [op for op in ("HASH_JOIN", "NL_JOIN", "MERGE_JOIN") if op in plan_text]
assert join_ops, plan_text
print("join operator:", join_ops[0])


**Check 2 - the pushdown that usually surprises people coming from row stores.** `ORDER BY ... LIMIT n`
does not have to sort the table; the planner folds it into a `TOP_N` operator.


In [ ]:
plan_text = run("EXPLAIN SELECT name, salary FROM employees ORDER BY salary DESC LIMIT 5")[0][1]
assert "TOP_N" in plan_text
print(plan_text)


`EXPLAIN ANALYZE` runs the query for real and attaches runtime numbers - rows and wall time per operator.
That is the measurement the plain plan only estimates.


In [ ]:
plan_text = run("EXPLAIN ANALYZE SELECT dept_id, COUNT(*) FROM employees GROUP BY dept_id")[0][1]
print(plan_text)

assert "Total Time" in plan_text and "Sequential Scan" in plan_text
print("runtime numbers attached to every operator")


**Approach and pitfalls.** Read plans top-down: the output operator first, the scans at the bottom.
`SEQ_SCAN` is DuckDB's bread and butter - a vectorised columnar scan is cheap, so unlike a b-tree OLTP
engine, a full scan of a narrow column set is often the best plan. A `FILTER` you find in the plan can
often be pushed further down; a `HASH_JOIN`'s build side should be the smaller table's. Two cautions:
`EXPLAIN ANALYZE` *executes* the statement (side effects included), and the plain `EXPLAIN` plan reflects
estimates - keep the plan and the measurement apart.


---
## Interview questions

Short theory questions that interviewers ask alongside the coding exercises. Each one is split into the question
and its model answer - say the answer out loud before you read the next cell.

### T1. What is an index, and what does a B-tree index actually do?

*Answer it out loud before reading on.*

An index is a separate, ordered data structure that maps key values to the physical location of rows. OLTP
engines default to a B-tree: a balanced, high-fanout tree that keeps keys sorted, so a lookup is O(log n) page
reads instead of a full table scan. Leaves are linked, which also makes range scans (`BETWEEN`, `>`,
`ORDER BY`) cheap.

The costs matter as much as the benefits: every index adds space and makes `INSERT`/`UPDATE`/`DELETE` slower
because the index must be maintained. Index the columns you filter and join on most often, and prefer composite
indexes whose column order matches the query's leading predicates (leftmost-prefix rule). An index on
`(dept_id, salary)` can serve `WHERE dept_id = ?` and `WHERE dept_id = ? ORDER BY salary`, but not
`WHERE salary = ?` alone. One footnote: DuckDB is columnar - it answers analytics directly from the columns
and keeps ART (trie) indexes only for point lookups, so the honest line is "B-trees in OLTP, almost none in a
columnar warehouse".


### T2. How do you read `EXPLAIN` / `EXPLAIN ANALYZE`, and what do you look for?

*Answer it out loud before reading on.*


`EXPLAIN` shows the physical plan the optimiser chose for a statement; add `ANALYZE` to run the query for real
and attach runtime numbers. The things to look for:

- `SEQ_SCAN`: the columnar sequential scan reads every row group. Normal for analytics and for small tables.
- `HASH_JOIN`: an equi-join built on a hash table.
- `HASH_GROUP_BY` / `PERFECT_HASH_GROUP_BY`: grouping - the perfect hash variant appears when the number of
  distinct keys is small.
- `TOP_N`: `ORDER BY ... LIMIT n` answered in a single pass instead of a full sort.
- `FILTER`: a post-scan predicate.

The plan is a *plan*, not a measurement: `EXPLAIN ANALYZE` (Part 7, skill 5) is what you measure with.


**Demo 1 - the plan for the duplicate-email query.** An unindexed `GROUP BY email` has no choice but a
sequential scan feeding a hash group-by, then a `FILTER` for the `HAVING`.


In [ ]:
plan_text = run("EXPLAIN " + """
    SELECT email, COUNT(*) FROM employees GROUP BY email HAVING COUNT(*) > 1
""")[0][1]
print(plan_text)
assert "SEQ_SCAN" in plan_text and "GROUP_BY" in plan_text


**Demo 2 - a filter with an index in place.** DuckDB keeps indexes as ART (trie) indexes for point lookups;
as a column store it usually answers analytics straight from the columns, which is why the plan below is
unremarkable - and why result checks, not plan shape, prove the index does not break anything.


In [ ]:
run("CREATE INDEX IF NOT EXISTS idx_employees_dept ON employees(dept_id, salary)")
show("SELECT name, salary FROM employees WHERE dept_id = 2 ORDER BY salary DESC")
assert run("SELECT name, salary FROM employees WHERE dept_id = 2 ORDER BY salary DESC") == [
    ("Frank", 130000),
    ("Grace", 98000),
    ("Heidi", 87000),
    ("Ivan", 87000),
]
print("indexed lookup results unchanged")


**Demo 3 - confirm the index exists.** `duckdb_indexes()` is the catalogue of every created index, which is
how you check what the setup actually built.


In [ ]:
assert run(
    "SELECT index_name FROM duckdb_indexes() WHERE index_name = 'idx_employees_dept'"
) == [("idx_employees_dept",)]
print("index registered")


### T3. What is normalisation? Explain 1NF, 2NF and 3NF, and when you would denormalise.

*Answer it out loud before reading on.*

Normalisation organises columns and tables to remove redundancy and update anomalies.

- **1NF**: every column holds a single atomic value (no repeating groups or comma-separated lists) and rows are
  unique.
- **2NF**: 1NF plus no partial dependency - every non-key column depends on the *whole* primary key, which only
  matters for composite keys.
- **3NF**: 2NF plus no transitive dependency - non-key columns depend on the key, not on another non-key column
  (for example storing `dept_name` next to `emp_id` when `dept_id` already determines it).

**BCNF** is a stricter 3NF for tables with overlapping candidate keys. You denormalise deliberately for read
performance or simplicity: star schemas keep a dimension's attributes together to avoid joins, and analytics
tables pre-join facts with dimensions. The trade-off is redundant storage and the risk of inconsistent updates,
which is why denormalised layers are usually rebuilt from a normalised source of truth.

**Follow-up.** Where does a star schema sit relative to 3NF? (Answer: a dimension is typically 2NF/3NF-ish, and
the fact table is intentionally denormalised.)

### T4. What does ACID mean, and what are the isolation levels?

*Answer it out loud before reading on.*

- **Atomicity**: all statements in a transaction commit or none do.
- **Consistency**: a transaction moves the database from one valid state to another (constraints hold).
- **Isolation**: concurrent transactions do not see each other's uncommitted work.
- **Durability**: once committed, data survives a crash.

The SQL standard defines four isolation levels, each permitting fewer anomalies:

| level | dirty read | non-repeatable read | phantom read |
| --- | --- | --- | --- |
| Read uncommitted | yes | yes | yes |
| Read committed | no | yes | yes |
| Repeatable read | no | no | yes |
| Serializable | no | no | no |

Most engines default to Read Committed (Postgres, Oracle) or Repeatable Read (MySQL InnoDB); embedded engines
serialise instead - DuckDB allows one writer transaction per database at a time. In practice you also meet
**snapshot isolation** and **write skew** (MVCC): Postgres' Repeatable Read is a snapshot isolation level that
prevents write skew by aborting conflicting transactions. Isolation is a correctness-versus-throughput dial:
stronger isolation means more locking or more aborts.


### T5. `GROUP BY` vs window functions: what is the difference?

*Answer it out loud before reading on.*

`GROUP BY` collapses rows: the result has one row per group, so you lose the individual rows. Window functions
keep every row and compute the aggregate (or ranking) across a defined window, which is why they can add
"department average" or "running total" as extra columns.

```sql
-- one row per department
SELECT dept_id, AVG(salary) FROM employees GROUP BY dept_id;

-- every employee keeps their row and gains the department average
SELECT name, salary, AVG(salary) OVER (PARTITION BY dept_id) FROM employees;
```

Windows are evaluated after `WHERE`, `GROUP BY` and `HAVING`, which is why you cannot filter on a window alias
in `WHERE` - use an outer query, or `QUALIFY`, which DuckDB, Snowflake and BigQuery support directly (Part 7,
skill 1). Window functions can also rank (`ROW_NUMBER`/`RANK`/`DENSE_RANK`) and access other rows
(`LAG`/`LEAD`, `FIRST_VALUE`), which `GROUP BY` cannot do without a self-join.


### T6. `UNION` vs `UNION ALL`

*Answer it out loud before reading on.*

`UNION` concatenates two result sets and removes duplicates, which requires a sort or hash and is therefore slower
and memory-hungry. `UNION ALL` simply concatenates, keeping duplicates. The rule of thumb: use `UNION ALL` unless
you specifically need deduplication, which is almost never in ETL where duplicates usually signal a bug rather
than something to hide. Both require compatible column counts and types; only the first query's column names are
used. `INTERSECT` and `EXCEPT` (minus) are the set-difference counterparts.

### T7. Explain `NULL` semantics, and the traps around `NOT IN` and `COUNT`.

*Answer it out loud before reading on.*

`NULL` means "unknown", not "empty". Any comparison with `NULL` yields `UNKNOWN`, and `WHERE` keeps only rows
where the predicate is `TRUE`. So `salary = NULL` and `salary <> NULL` both return nothing; use `IS NULL` /
`IS NOT NULL`. Three-valued logic propagates through `AND`/`OR`: `TRUE OR NULL` is `TRUE`, but `TRUE AND NULL` is
`NULL`.

Traps:

- `NOT IN (subquery)` returns **no rows** if the subquery yields even one `NULL`, because `x <> NULL` is unknown.
  Prefer `NOT EXISTS`.
- `COUNT(*)` counts rows; `COUNT(col)` counts non-null values; `COUNT(DISTINCT col)` ignores nulls too.
- `SUM`/`AVG`/`MIN`/`MAX` skip nulls, so `AVG` over a column with nulls divides by the non-null count. `SUM` of
  no rows is `NULL`, not 0 - wrap it in `COALESCE`.
- `NULL = NULL` is unknown, so joins on nullable keys behave surprisingly.
- Many engines index nulls (Oracle does not by default), and `GROUP BY` treats all nulls as one group.

### T8. `HAVING` vs `WHERE`

*Answer it out loud before reading on.*

`WHERE` filters individual rows *before* grouping and aggregation; `HAVING` filters groups *after* aggregation.
`WHERE status = 'completed'` removes rows, `HAVING COUNT(*) > 1` removes whole groups. Because of that order,
`HAVING` may reference aggregates and `WHERE` may not; conversely, filtering in `WHERE` is usually cheaper
because fewer rows reach the aggregation step. A common interview follow-up is "can you use an aggregate in
`WHERE`?" - no, and the fix is either `HAVING` or a subquery/CTE that computes the aggregate first.

### T9. CTE vs subquery: when do you use each?

*Answer it out loud before reading on.*

A CTE (`WITH name AS (...)`) is a named subquery. It improves readability, lets you reference the same result
more than once, and can be recursive. Whether it is materialised or inlined depends on the engine: Postgres 12+
inlines non-recursive CTEs unless marked `MATERIALIZED`; DuckDB, BigQuery and Snowflake treat them as named
subqueries the optimiser may inline. Historically, CTEs were an "optimisation fence" in older Postgres, which is
why the old advice was "CTEs are slow" - measure your engine rather than assume. Use a CTE for clarity or
reuse; use a plain subquery for a one-off inline filter. For hierarchies (org charts, bill of materials, graph
traversal), a recursive CTE is the tool:

```sql
WITH RECURSIVE subordinates AS (
    SELECT emp_id, manager_id FROM employees WHERE emp_id = 1
    UNION ALL
    SELECT e.emp_id, e.manager_id
    FROM employees e JOIN subordinates s ON e.manager_id = s.emp_id
)
SELECT * FROM subordinates;
```


### T10. Star vs snowflake schema

*Answer it out loud before reading on.*

A **star schema** has one central fact table (one row per business event, with foreign keys and measures) joined
directly to denormalised dimension tables. It is simple, requires few joins, and is what most BI tools assume. A
**snowflake schema** normalises dimensions into sub-dimensions (product -> category -> department), saving
storage and removing redundancy but requiring more joins.

In practice, star wins for analytics because storage is cheap and joins cost time; snowflaking is worth it when a
dimension is enormous or shared across many marts. The related idea is **one big table (OBT)** / wide denormalised
tables, which push the join cost to the transformation layer. Fact-table design also distinguishes
transaction facts (one row per event), periodic snapshots (one row per entity per period) and accumulating
snapshots (one row per process instance, updated as it moves through stages).

### T11. What is a slowly changing dimension (SCD), and how does Type 2 work?

*Answer it out loud before reading on.*

An SCD is a dimension attribute that changes over time rather than staying fixed. The types:

- **Type 0**: never changes (a true natural key).
- **Type 1**: overwrite in place - no history, simplest, loses the old value.
- **Type 2**: add a new row per version, with `valid_from`/`valid_to` (or `start_date`/`end_date`) and often a
  `is_current` flag and a new surrogate key. History is preserved and facts can be joined to the version that
  was current when the event happened.
- **Type 3**: keep a limited number of previous values as extra columns (`current_region`, `prior_region`).
- **Type 6**: a hybrid of 1, 2 and 3.

Type 2 is the standard interview answer. The implementation closes the current row (`valid_to = today`,
`is_current = false`) and inserts a new row with `valid_from = today`. Getting the boundaries right matters:
`valid_to` is usually exclusive (`[valid_from, valid_to)`), and a common bug is overlapping or gapped ranges.
Loading Type 2 with a merge/incremental model (`dbt snapshot` is the classic tool) is the day-to-day reality.

### T12. What is partitioning, and how does it help?

*Answer it out loud before reading on.*

Partitioning physically splits a large table into smaller pieces by a key - usually a date range, or a hash or
list of keys - while queries still see one logical table. The main win is **partition pruning**: a query filtered
on the partition key reads only the relevant partitions instead of the whole table. It also enables partition-wise
maintenance (drop an old partition instead of `DELETE`ing rows) and parallel scans.

Things to know: the partition key should appear in the `WHERE` clause of the important queries, or pruning cannot
happen; too many small partitions hurt (metadata overhead); and partitioning is not the same as clustering or
bucketing, which order rows *within* a partition. Many warehouses combine both: partition by day, cluster by
customer id.

### T13. How do you approach query optimisation?

*Answer it out loud before reading on.*

A repeatable checklist beats guessing:

1. **Measure first.** Get the actual plan and timings, not a hunch.
2. **Read the plan** (`EXPLAIN` / `EXPLAIN ANALYZE`; Part 7, skill 5 walks a DuckDB plan operator by operator -
   `SEQ_SCAN`, `HASH_JOIN`, `HASH_GROUP_BY`, `TOP_N`). Look for full scans on big tables, bad join orders, and
   spills/sorts.
3. **Reduce rows early.** Push `WHERE` predicates down, filter before joining, and select only the columns you
   need - `SELECT *` blocks covering indexes and moves data you will never read.
4. **Index for the access pattern** you actually measured: composite indexes matching the leading predicates and
   the `ORDER BY`; avoid indexing low-cardinality columns on their own.
5. **Fix the join strategy** where needed: statistics, correct data types, and explicit join keys.
6. **Pre-aggregate or materialise** expensive repeated work; incremental models instead of full rebuilds.
7. **Partition and prune** large tables.
8. **Verify with the new plan and timings**, and watch the write cost you added.

Common anti-patterns: functions on the indexed column (`WHERE date(created_at) = ...`), implicit type casts,
`SELECT DISTINCT` to paper over a bad join, and `NOT IN` with nullable subqueries.


### T14. Clustered vs non-clustered indexes, and what is a covering index?

*Answer it out loud before reading on.*

A **clustered** index defines the physical order of the table's rows - there can be only one, and in InnoDB and
SQL Server the primary key is clustered by default. Because rows are stored in key order, range scans on the
clustered key are very fast, and the index *is* the data (no extra lookup). A **non-clustered** index is a
separate structure: it stores the key plus a pointer (row id / primary key) back to the row, so a lookup may need
a second read, called a **key lookup** or **bookmark lookup**.

A **covering index** contains all the columns a query needs, so the engine can answer it from the index alone -
this is the "index-only scan". For example, an index on `(dept_id, salary)` covers
`SELECT salary FROM employees WHERE dept_id = ?` without touching the table. In Postgres you make this explicit
with `INCLUDE` columns, and in InnoDB the primary key is implicitly appended to every secondary index, which is
why keeping the primary key narrow matters. DuckDB is the outlier: as a column store its scans already read only
the referenced columns, and its indexes are ART tries for point lookups rather than covering structure.
Trade-offs are the same as any index: extra storage and slower writes.


---
## Where to go next

You have now seen the 22 coding questions, the five DuckDB-only skills (`QUALIFY`, `PIVOT`/`UNPIVOT`,
`ASOF JOIN`, typed `LIST`/`STRUCT`, readable `EXPLAIN`/`EXPLAIN ANALYZE`) and the 14 theory questions. To make
the practice stick: re-run the notebook after changing the seed data (add a third duplicate email, or a customer
with only cancelled orders) and see which assertions fail - that is exactly the edge-case thinking an
interviewer is probing for. Run it live from the viewer too: `make dev-local`, then open
`/topics/03_sql_interview_questions` and execute cells against the persistent Python kernel.
